# El mapa de lo que no se mide
**Concurso Contar con Datos (UdeSA) · Exploración interactiva**

Este notebook consulta los metadatos **reales** de [Datos Argentina](https://datos.gob.ar) mediante la API CKAN del portal (el viejo `data.json` ya no existe desde la migración a CKAN 2.11), analiza cada conjunto de datos y genera un tablero HTML interactivo con:

- una **introducción guiada por scroll** que cuenta los hallazgos principales sobre una nube 3D de los conjuntos;
- el **estado de actualización** de cada conjunto (al día, atrasado o abandonado) comparado con la frecuencia que prometió su propio organismo, y cuántas **actualizaciones prometidas nunca llegaron**;
- la **nube del olvido** (3D, Three.js): ángulo = organismo o tema, distancia al centro = años sin actualizar, altura = actualizaciones que faltan;
- **la ciudad del olvido** (3D): una manzana por organismo o tema y un edificio por conjunto, con altura = años sin actualizar; con un botón se cambia al **mosaico 2D** con zoom;
- el año en que dejaron de actualizarse (un punto por conjunto, con animación de caída), un ranking de organismos y una tabla ordenable con buscador propio;
- una **ficha** por conjunto con la línea de tiempo de sus actualizaciones perdidas, un buscador que lleva la cámara hasta el conjunto y filtros que quedan guardados en la dirección de la página.

**Cómo usarlo:** ejecutá todo con *Entorno de ejecución → Ejecutar todo*. Al final se descargan `mapa_de_lo_que_no_se_mide.html` y `datasets_analizados.csv`. Necesita conexión a internet (Colab ya la tiene). El HTML carga Three.js y las tipografías desde CDN, así que también necesita internet para verse completo.

Si después de la celda 3 aparece un aviso sobre campos no encontrados, copiá la lista de campos que imprime y ajustá los patrones de `buscar()`.

In [ ]:
# ── Configuración ───────────────────────────────────────────────────────────
import json, re, time, unicodedata

import pandas as pd
import requests

# Desde 2026 Datos Argentina corre sobre CKAN 2.11 (el viejo /data.json ya no existe).
API_URL = "https://datos.gob.ar/api/3/action/package_search"
PORTAL  = "https://datos.gob.ar"
SALIDA_HTML = "mapa_de_lo_que_no_se_mide.html"
SALIDA_CSV  = "datasets_analizados.csv"

AHORA = pd.Timestamp.now(tz="UTC")
print("Fecha de corte del análisis:", AHORA.date())


In [ ]:
# ── Descarga de los metadatos reales (API CKAN de datos.gob.ar) ─────────────
# El servidor corta las respuestas grandes ("Response ended prematurely"), así que
# se pide en tandas chicas y, si una falla, se achica la tanda y se reintenta.
def descargar_catalogo(url=API_URL, por_pagina=50, minimo=5, pausa=0.3, max_fallos=12):
    headers = {"User-Agent": "Mozilla/5.0 (Colab; concurso Contar con Datos)", "Accept": "application/json"}
    sesion = requests.Session()
    vistos, resultados = set(), []
    inicio, total, fallos, usar_sort = 0, None, 0, True

    while total is None or inicio < total:
        params = {"rows": por_pagina, "start": inicio}
        if usar_sort:
            params["sort"] = "name asc"          # orden estable para paginar
        try:
            r = sesion.get(url, params=params, headers=headers, timeout=(15, 90))
            r.raise_for_status()
            data = r.json()
            if not data.get("success"):
                raise RuntimeError(data.get("error"))
        except Exception as e:
            fallos += 1
            if fallos > max_fallos:
                raise RuntimeError(f"Demasiados errores consultando {url}. Último: {e}")
            if isinstance(e, requests.HTTPError) and usar_sort and e.response is not None and e.response.status_code in (400, 409, 500):
                usar_sort = False                  # por si el portal no acepta 'sort'
                print("  el portal rechazó 'sort'; reintento sin orden")
            else:
                por_pagina = max(minimo, por_pagina // 2)
            print(f"  falló (start={inicio}): {type(e).__name__}. Reintento con tandas de {por_pagina}…")
            time.sleep(2 * fallos)
            continue

        res = data["result"]
        total = res["count"]
        if not res["results"]:
            break
        for d in res["results"]:
            if d.get("id") not in vistos:
                vistos.add(d.get("id"))
                resultados.append(d)
        inicio += len(res["results"])
        print(f"  {len(resultados):,} de {total:,} conjuntos", end="\r")
        time.sleep(pausa)
    print()
    return resultados

def extraer_datasets(obj):
    """Acepta la respuesta de package_search o una lista de datasets."""
    if isinstance(obj, list):
        return obj
    if isinstance(obj, dict) and "result" in obj and "results" in obj["result"]:
        return obj["result"]["results"]
    return []

try:
    datasets = descargar_catalogo()
except Exception as e:
    # Plan B: guardá desde el navegador respuestas de
    #   https://datos.gob.ar/api/3/action/package_search?rows=100&start=0   (y start=100, 200, … hasta cubrir el total)
    # y subí todos los archivos juntos acá.
    print(e)
    print("Subí los archivos JSON guardados desde el navegador:")
    from google.colab import files
    subidos = files.upload()
    datasets = []
    for contenido in subidos.values():
        datasets += extraer_datasets(json.loads(contenido.decode("utf-8-sig")))

print(f"Conjuntos de datos obtenidos: {len(datasets):,}")


In [ ]:
# ── Procesamiento: antigüedad, estado, completitud y cobertura territorial ──
from collections import Counter

def s(x):
    return x.strip() if isinstance(x, str) else ("" if x is None else str(x).strip())

def norm(txt):
    txt = unicodedata.normalize("NFD", txt or "")
    return "".join(c for c in txt if unicodedata.category(c) != "Mn").lower()

def fecha(x):
    f = pd.to_datetime(s(x), errors="coerce", utc=True)
    return None if pd.isna(f) else f

def frecuencia_dias(txt):
    """'R/P1Y (anualmente)', 'R/P1M', 'R/P3M'… (ISO 8601) → días entre actualizaciones. None si es eventual o falta."""
    txt = s(txt).upper()
    if not txt or "EVENTUAL" in txt:
        return None
    t = txt.split()[0].replace("R/", "")
    m = re.fullmatch(r"P(?:(\d+(?:\.\d+)?)Y)?(?:(\d+(?:\.\d+)?)M)?(?:(\d+(?:\.\d+)?)W)?(?:(\d+(?:\.\d+)?)D)?"
                     r"(?:T(?:(\d+(?:\.\d+)?)H)?(?:(\d+(?:\.\d+)?)M)?(?:(\d+(?:\.\d+)?)S)?)?", t)
    if not m:
        return None
    y, mo, w, d, h, mi, sg = [float(v) if v else 0.0 for v in m.groups()]
    dias = y * 365 + mo * 30.4 + w * 7 + d + h / 24 + mi / 1440 + sg / 86400
    return dias if dias > 0 else None

def etiqueta_frecuencia(txt, dias):
    if "EVENTUAL" in s(txt).upper():
        return "Eventual"
    if dias is None:
        return "No declarada"
    if dias < 0.9:
        return "Varias veces por día"
    for ref, nombre in [(1, "Diaria"), (7, "Semanal"), (15, "Quincenal"), (30.4, "Mensual"), (60.8, "Bimestral"),
                        (91.2, "Trimestral"), (121.6, "Cuatrimestral"), (182.4, "Semestral"), (365, "Anual")]:
        if abs(dias - ref) / ref < 0.1:
            return nombre
    if dias > 365 and abs(dias / 365 - round(dias / 365)) < 0.05:
        return f"Cada {round(dias / 365)} años"
    return f"Cada {dias:g} días"

def estado_de(antig_dias, frec_dias):
    """Al día / Atrasado / Abandonado según lo que el propio organismo prometió."""
    if antig_dias is None:
        return "Sin fecha"
    if frec_dias is None:
        return "Abandonado" if antig_dias > 3 * 365 else "Sin periodicidad"
    tolerancia = max(1.5 * frec_dias, 14)      # hasta 1,5 períodos (mínimo 14 días) se considera al día
    limite     = max(4.0 * frec_dias, 90)      # más de 4 períodos (mínimo 90 días) = abandonado
    if antig_dias <= tolerancia:
        return "Al día"
    return "Atrasado" if antig_dias <= limite else "Abandonado"

def actualizaciones_faltantes(antig_dias, frec_dias, estado):
    """Cuántas veces debió actualizarse desde la última vez (antigüedad ÷ período, entero).
    Se cuenta como máximo una por día aunque la ficha prometa cada hora o cada segundo."""
    if frec_dias is None or antig_dias is None or estado == "Al día":
        return 0
    return int(antig_dias // max(frec_dias, 1))

# Provincias: (nombre, lat, lon, patrón). Las ambiguas exigen "provincia de X".
PROVINCIAS = [
    ("CABA", -34.61, -58.44, r"\bcaba\b|ciudad autonoma de buenos aires|ciudad de buenos aires"),
    ("Buenos Aires", -36.6, -60.0, r"provincia de buenos aires|\bpba\b|bonaerense"),
    ("Catamarca", -27.3, -66.9, r"catamarca"),
    ("Chaco", -26.4, -60.8, r"\bchaco\b"),
    ("Chubut", -43.8, -68.9, r"chubut"),
    ("Córdoba", -31.4, -63.9, r"cordoba"),
    ("Corrientes", -28.7, -57.8, r"provincia de corrientes|provincia corrientes"),
    ("Entre Ríos", -32.0, -59.2, r"entre rios"),
    ("Formosa", -25.2, -59.9, r"provincia de formosa|\bformosa\b"),
    ("Jujuy", -23.3, -65.8, r"jujuy"),
    ("La Pampa", -37.1, -65.4, r"la pampa"),
    ("La Rioja", -29.8, -67.4, r"la rioja"),
    ("Mendoza", -34.6, -68.4, r"mendoza"),
    ("Misiones", -26.9, -54.6, r"provincia de misiones|provincia misiones"),
    ("Neuquén", -38.6, -70.1, r"neuquen"),
    ("Río Negro", -40.0, -67.8, r"provincia de rio negro|\brio negro\b"),
    ("Salta", -24.3, -64.8, r"provincia de salta|\bsalta,|\bsalta\b(?= (capital|y jujuy))"),
    ("San Juan", -30.9, -68.9, r"provincia de san juan|\bsan juan\b"),
    ("San Luis", -33.7, -66.0, r"\bsan luis\b"),
    ("Santa Cruz", -48.8, -70.2, r"santa cruz"),
    ("Santa Fe", -31.0, -60.9, r"santa fe"),
    ("Santiago del Estero", -27.8, -63.3, r"santiago del estero"),
    ("Tierra del Fuego", -54.3, -67.7, r"tierra del fuego"),
    ("Tucumán", -26.9, -65.3, r"tucuman"),
]
PROV_RE = [(n, re.compile(p)) for n, _, _, p in PROVINCIAS]

def provincias_en(texto):
    t = norm(texto)
    return [n for n, rx in PROV_RE if rx.search(t)]

# Códigos de provincia del INDEC (así vienen los campos 'province' y 'district' del portal).
CODIGOS_INDEC = {"02": "CABA", "06": "Buenos Aires", "10": "Catamarca", "14": "Córdoba", "18": "Corrientes",
                 "22": "Chaco", "26": "Chubut", "30": "Entre Ríos", "34": "Formosa", "38": "Jujuy", "42": "La Pampa",
                 "46": "La Rioja", "50": "Mendoza", "54": "Misiones", "58": "Neuquén", "62": "Río Negro", "66": "Salta",
                 "70": "San Juan", "74": "San Luis", "78": "Santa Cruz", "82": "Santa Fe", "86": "Santiago del Estero",
                 "90": "Tucumán", "94": "Tierra del Fuego"}

def provincia_estructurada(valor, distrito=""):
    """Campo 'province' (código INDEC como '06', o nombre) o, si falta, los 2 primeros dígitos de 'district' → provincia."""
    v = s(valor)
    if v.isdigit() and len(v) <= 2:
        return [CODIGOS_INDEC[v.zfill(2)]] if v.zfill(2) in CODIGOS_INDEC else []
    n = norm(v)
    if n and n != "none":
        if re.search(r"\bcaba\b|ciudad autonoma|ciudad de buenos aires|capital federal", n):
            return ["CABA"]
        if n.startswith("tierra del fuego"):
            return ["Tierra del Fuego"]
        if n.strip() in ("buenos aires", "provincia de buenos aires", "pba"):
            return ["Buenos Aires"]
        hallados = [nombre for nombre, *_ in PROVINCIAS if norm(nombre) in n and nombre not in ("CABA", "Buenos Aires")]
        if hallados:
            return hallados
    d_ = s(distrito)
    if d_.isdigit() and len(d_) >= 5 and d_[:2] in CODIGOS_INDEC:
        return [CODIGOS_INDEC[d_[:2]]]
    return []

def campos_planos(d):
    """Campos escalares del dataset + 'extras' de CKAN en un único diccionario clave→texto."""
    c = {k: s(v) for k, v in d.items() if isinstance(v, (str, int, float)) and s(v) != ""}
    for e in d.get("extras") or []:
        if isinstance(e, dict) and e.get("key"):
            c.setdefault(e["key"], s(e.get("value")))
    return c

def buscar(c, patron):
    """Primer valor cuya clave (sin tildes, en minúsculas) coincide con el patrón."""
    rx = re.compile(patron)
    for k, v in c.items():
        if v and rx.search(norm(k)):
            return v
    return ""

def titulos(lista, *claves):
    out = []
    for v in lista or []:
        if isinstance(v, dict):
            out.append(next((s(v[k]) for k in claves if v.get(k)), ""))
        else:
            out.append(s(v))
    return [x for x in out if x]

def codigo_tema(d):
    v = d.get("dataset_superTheme")
    v = v[0] if isinstance(v, list) and v else v
    return s(v).rstrip("/").split("/")[-1].upper()

# Los conjuntos sin 'groups' pero con código de supertema (ej. ENVI) toman el nombre que ese código tiene en el resto del catálogo.
_cnt = {}
for d in datasets:
    g = titulos(d.get("groups"), "display_name", "title", "name")
    if g and codigo_tema(d):
        _cnt.setdefault(codigo_tema(d), Counter())[g[0]] += 1
tema_por_codigo = {k: v.most_common(1)[0][0] for k, v in _cnt.items()}

filas = []
for i, d in enumerate(datasets):
    c = campos_planos(d)
    org = d.get("organization") or {}
    organismo = s(org.get("title") if isinstance(org, dict) else org) or buscar(c, r"publicador|publisher") or "Sin organismo declarado"

    grupos = titulos(d.get("groups"), "display_name", "title", "name")
    cod = codigo_tema(d)
    tema = grupos[0] if grupos else (tema_por_codigo.get(cod) or cod or "Sin tema")

    recursos = [r for r in (d.get("resources") or []) if isinstance(r, dict)]
    f_datos = [fecha(r.get(k)) for r in recursos for k in ("last_modified", "modified", "created")]
    f_datos += [fecha(v) for k, v in c.items() if ("modif" in norm(k) or "actualizados al" in norm(k)) and not k.startswith("metadata_")]
    f_datos = [f for f in f_datos if f is not None]
    # 'metadata_modified' NO se usa: en el portal nuevo es la fecha de la migración y es igual para todos los conjuntos.
    ultima = max(f_datos) if f_datos else None
    antig = max((AHORA - ultima).days, 0) if ultima is not None else None

    per_txt = buscar(c, r"accrual|periodic|frecuen|frequency")
    fdias = frecuencia_dias(per_txt)
    estado = estado_de(antig, fdias)

    keywords = titulos(d.get("tags"), "display_name", "name")
    titulo = s(d.get("title")) or "(sin título)"
    desc = s(d.get("notes"))
    spatial = buscar(c, r"spatial|espacial|geograf")
    temporal = buscar(c, r"temporal")
    con_descarga = any(s(r.get("url")) for r in recursos)
    licencia = s(d.get("license_title") or d.get("license_id"))

    completitud = (20 * (len(desc) >= 40) + 10 * bool(grupos) + 10 * bool(keywords) + 15 * bool(per_txt)
                   + 15 * bool(spatial or temporal) + 10 * bool(licencia) + 20 * con_descarga)

    provs = provincia_estructurada(buscar(c, r"^(province|provincia)$"), buscar(c, r"^district$"))
    for p_ in provincias_en(" ".join([titulo, desc, " ".join(keywords), spatial])):
        if p_ not in provs:
            provs.append(p_)

    filas.append(dict(
        i=i, id=s(d.get("name") or d.get("id")), t=titulo, p=organismo, th=tema, st=estado,
        d=ultima.strftime("%Y-%m-%d") if ultima is not None else None,
        age=round(antig / 365.25, 2) if antig is not None else None,
        f=etiqueta_frecuencia(per_txt, fdias), pr=int(fdias is not None),
        fd=round(fdias, 6) if fdias else None, mi=actualizaciones_faltantes(antig, fdias, estado),
        c=int(completitud), pv=provs, n=len(recursos),
        u=f"{PORTAL}/dataset/{s(d.get('name'))}" if d.get("name") else "",
    ))

df = pd.DataFrame(filas)

# ── Diagnóstico: ¿se encontraron los campos clave? ──
print(f"{len(df):,} conjuntos procesados\n")
print(f"Con frecuencia de actualización detectada: {df.pr.mean():.0%}")
print(f"Con fecha de actualización:                {df.d.notna().mean():.0%}")
print(f"Con alguna provincia mencionada:           {(df.pv.str.len() > 0).mean():.0%}")
print(f"Con tema asignado:                         {(df.th != 'Sin tema').mean():.0%}")
print("Frecuencias declaradas:", df.f.value_counts().head(8).to_dict())
if df.pr.mean() < 0.2 and len(datasets):
    print("\n⚠ Casi ningún conjunto declara frecuencia. Probablemente los campos tengan otro nombre.")
    print("  Campos del primer conjunto, para ajustar los patrones de buscar():")
    print(sorted(campos_planos(datasets[0]).keys()))
df["st"].value_counts()


In [ ]:
# ── Hallazgos rápidos (para tu relato del concurso) ─────────────────────────
total = len(df)
con_promesa = df[df.pr == 1]
incumplen = con_promesa[con_promesa.st.isin(["Atrasado", "Abandonado"])]

print(f"Conjuntos analizados: {total:,}")
print(f"Declaran una frecuencia de actualización: {len(con_promesa):,} ({len(con_promesa)/total:.0%})")
print(f"  …de ésos, incumplen su propia promesa: {len(incumplen):,} ({len(incumplen)/max(len(con_promesa),1):.0%})")
print(f"Actualizaciones prometidas que nunca llegaron: {df.mi.sum():,}")
print(f"Mediana de años sin actualizar: {df.age.median():.1f}")
print("\nDistribución por estado:")
print((df.st.value_counts(normalize=True) * 100).round(1).astype(str) + " %")

por_org = (df.groupby("p").agg(n=("id", "size"), abandonados=("st", lambda x: (x == "Abandonado").sum()))
             .query("n >= 5").assign(pct=lambda x: (100 * x.abandonados / x.n).round(0))
             .sort_values(["pct", "n"], ascending=False).head(10))
print("\nConjuntos a los que más actualizaciones les faltan:")
print(df.nlargest(5, "mi")[["t", "p", "f", "age", "mi"]].to_string(index=False))
print("\nOrganismos (≥5 conjuntos) con mayor % de abandonados:")
print(por_org)

cuenta_prov = {n: sum(n in v for v in df.pv) for n, *_ in PROVINCIAS}
print("\nProvincias con menos conjuntos que las mencionan:")
print(pd.Series(cuenta_prov).sort_values().head(8))


### Plantilla del tablero
Esta celda solo guarda el diseño HTML/CSS/JS del tablero (intro con scroll, nube 3D y ciudad 3D con Three.js, mosaico 2D, años, ranking, tabla y fichas). No hace falta tocarla.

In [ ]:
PLANTILLA_HTML = r"""<!doctype html>
<html lang="es">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>El mapa de lo que no se mide · Datos Argentina</title>
<meta name="description" content="Qué pasó con la promesa de actualizar los conjuntos de datos abiertos del Estado nacional argentino.">
<link rel="preconnect" href="https://fonts.googleapis.com">
<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,600;9..144,700&family=Public+Sans:wght@400;500;600;700&display=swap" rel="stylesheet">
<script type="importmap">{"imports":{"three":"https://cdn.jsdelivr.net/npm/three@0.160.0/build/three.module.js","three/addons/":"https://cdn.jsdelivr.net/npm/three@0.160.0/examples/jsm/"}}</script>
<style>
:root{
  color-scheme:dark;
  --bg:#0A101C; --panel:#111A2A; --panel2:#172338; --ink:#E7EDF4; --muted:#94A3B6; --faint:#5F6F85; --line:#22314A;
  --ok:#35B8A8; --late:#E9A23B; --dead:#E0507A; --nop:#9AA6B8; --nod:#4E5A6C;
  --serif:'Fraunces',Georgia,serif; --sans:'Public Sans',system-ui,-apple-system,'Segoe UI',sans-serif;
  --ease:cubic-bezier(.2,.8,.2,1);
}
*{box-sizing:border-box}
html{background:var(--bg);scroll-behavior:smooth}
body{margin:0;background:var(--bg);color:var(--ink);font:15px/1.55 var(--sans);-webkit-font-smoothing:antialiased}
button,select,input{font:inherit;color:inherit}
:focus-visible{outline:2px solid var(--ink);outline-offset:2px}
a{color:var(--ink)}
b,strong{font-weight:600}
.small{font-size:12.5px;color:var(--muted)}
.dot{display:inline-block;width:9px;height:9px;border-radius:50%;margin-right:6px;vertical-align:baseline}
.k-ok{color:var(--ok)} .k-late{color:var(--late)} .k-dead{color:var(--dead)} .k-nop{color:var(--nop)}
.gl{position:absolute;inset:0}
.gl canvas{display:block;width:100%!important;height:100%!important}

/* ── intro ── */
#story{position:relative}
#stage{position:sticky;top:0;height:100vh;height:100svh;overflow:hidden;
  background:radial-gradient(ellipse 80% 70% at 60% 55%,#15233D 0%,#0D1628 45%,#0A101C 80%)}
#stage::after{content:'';position:absolute;inset:0;pointer-events:none;z-index:1;background:linear-gradient(90deg,rgba(10,16,28,.82) 0%,rgba(10,16,28,.35) 34%,rgba(10,16,28,0) 52%)}
#steps{position:relative;z-index:2;margin-top:-100vh;margin-top:-100svh;pointer-events:none}
.step{min-height:100vh;min-height:100svh;display:flex;align-items:center;padding:0 6vw}
.step .card{pointer-events:auto;max-width:430px;background:rgba(10,16,28,.74);-webkit-backdrop-filter:blur(10px);backdrop-filter:blur(10px);
  border:1px solid var(--line);border-radius:14px;padding:22px 24px;opacity:.25;transform:translateY(16px);transition:opacity .6s,transform .6s var(--ease)}
.step.on .card{opacity:1;transform:none}
.step .card h2{font:600 clamp(22px,2.4vw,30px)/1.18 var(--serif);margin:0 0 10px;text-wrap:balance}
.step .card p{margin:0;color:#C3CEDB}
.step .card p b{color:var(--ink)}
.step.title .card{background:none;border:0;-webkit-backdrop-filter:none;backdrop-filter:none;max-width:700px;padding:0}
.step.title h1{font:700 clamp(40px,6.4vw,84px)/1.02 var(--serif);letter-spacing:-.02em;margin:10px 0 18px;text-wrap:balance}
.step.title p{font-size:clamp(16px,1.5vw,19px);max-width:46ch}
.kicker{font-size:12px;letter-spacing:.12em;text-transform:uppercase;color:var(--muted);margin:0}
.cue{margin-top:28px;font-size:13px;color:var(--muted);animation:bob 2.2s ease-in-out infinite}
@keyframes bob{50%{transform:translateY(6px)}}
.step:last-child{padding-bottom:10vh}
.cta{display:inline-block;margin-top:18px;padding:11px 18px;border-radius:999px;background:var(--ink);color:var(--bg);font-weight:600;text-decoration:none}
.topbar{position:absolute;top:0;left:0;right:0;z-index:3;display:flex;justify-content:space-between;align-items:center;padding:16px 6vw;font-size:13px}
.topbar .brand{font:600 15px var(--serif);color:var(--ink)}
.topbar a{color:var(--muted);text-decoration:none;border:1px solid var(--line);border-radius:999px;padding:6px 12px;background:rgba(10,16,28,.5)}
.topbar a:hover{color:var(--ink);border-color:var(--muted)}
#dots{position:absolute;right:22px;top:50%;transform:translateY(-50%);z-index:3;display:flex;flex-direction:column;gap:9px}
#dots i{width:7px;height:7px;border-radius:50%;background:var(--faint);transition:all .3s}
#dots i.on{background:var(--ink);transform:scale(1.4)}

/* ── tablero ── */
.shell{display:grid;grid-template-columns:268px minmax(0,1fr);gap:44px;max-width:1400px;margin:0 auto;padding:56px 28px 64px}
aside.side{position:sticky;top:20px;align-self:start;max-height:calc(100vh - 40px);overflow:auto}
#filtros summary{font:600 15px var(--sans);cursor:pointer;list-style:none;margin-bottom:14px}
#filtros summary::-webkit-details-marker{display:none}
.fbody{display:flex;flex-direction:column;gap:16px}
label{display:flex;flex-direction:column;gap:6px;font-size:13px;color:var(--muted)}
select,input[type=search]{width:100%;padding:9px 10px;border:1px solid var(--line);border-radius:8px;background:var(--panel);color:var(--ink)}
select option{background:var(--panel)}
input[type=range]{width:100%;accent-color:var(--ok)}
.sbox{position:relative}
#sug{position:absolute;z-index:20;left:0;right:0;top:calc(100% + 4px);margin:0;padding:4px;list-style:none;background:var(--panel2);border:1px solid var(--line);border-radius:8px;box-shadow:0 12px 30px rgba(0,0,0,.45)}
#sug li{padding:7px 8px;border-radius:6px;cursor:pointer;color:var(--ink);font-size:13px;line-height:1.3}
#sug li small{display:block;color:var(--muted);font-size:11.5px;margin-top:1px}
#sug li[aria-selected=true],#sug li:hover{background:#22314A}
.btn{padding:9px 13px;border:1px solid var(--line);border-radius:8px;background:var(--panel);cursor:pointer;font-weight:500;color:var(--ink);text-decoration:none;display:inline-flex;align-items:center;gap:6px;font-size:13.5px}
.btn:hover{border-color:var(--muted)}
.btn.primary{background:var(--ink);color:var(--bg);border-color:var(--ink)}
.seg{display:inline-flex;border:1px solid var(--line);border-radius:8px;overflow:hidden;background:var(--panel)}
.seg button{padding:6px 11px;border:0;background:none;cursor:pointer;font-size:12.5px;color:var(--muted)}
.seg button[aria-pressed=true]{background:var(--ink);color:var(--bg)}
.chip{display:inline-flex;gap:8px;align-items:center;background:var(--panel2);border:1px solid var(--line);border-radius:999px;padding:3px 6px 3px 11px;font-size:12.5px;cursor:pointer}

main h1{font:700 clamp(28px,3.4vw,44px)/1.1 var(--serif);letter-spacing:-.01em;margin:8px 0 22px;max-width:24ch;text-wrap:balance}
main h1 .num{color:var(--dead);font-variant-numeric:tabular-nums}
.kpis{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px;margin-bottom:26px}
.kpi{background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:14px 16px}
.kpi b{display:block;font:600 clamp(24px,2.4vw,32px)/1.1 var(--serif);font-variant-numeric:tabular-nums}
.kpi span{display:block;font-size:12.5px;color:var(--muted);margin-top:6px;line-height:1.35}
#bar{display:flex;height:42px;border-radius:8px;overflow:hidden;gap:2px}
#bar button{border:0;cursor:pointer;min-width:0;padding:0 10px;text-align:left;color:#0A101C;font-weight:700;font-size:13px;white-space:nowrap;overflow:hidden;transition:opacity .2s,flex-grow .6s var(--ease)}
#bar button.off{opacity:.2}
#legend{display:flex;flex-wrap:wrap;gap:6px 18px;margin:10px 0 0;font-size:13px;color:var(--muted)}
section.blk{border-top:1px solid var(--line);margin-top:48px;padding-top:22px}
section.blk h2{font:700 24px/1.2 var(--serif);margin:0 0 6px}
.lede{margin:0 0 16px;color:var(--muted);max-width:74ch}
.lede b{color:var(--ink)}
.sechead{display:flex;justify-content:space-between;align-items:flex-end;gap:16px;flex-wrap:wrap;margin-bottom:12px}
.sechead .lede{margin:0}

/* nube 3D */
.cloudwrap{position:relative;height:clamp(480px,74vh,740px);border-radius:16px;overflow:hidden;border:1px solid var(--line);
  background:radial-gradient(ellipse 75% 70% at 50% 55%,#15233D 0%,#0D1628 50%,#0A101C 85%);touch-action:none}
.cui{position:absolute;z-index:3;display:flex;gap:8px;flex-wrap:wrap;align-items:center}
.cui.tl{top:14px;left:14px} .cui.tr{top:14px;right:14px;justify-content:flex-end}
.cui.bl{bottom:14px;left:14px;font-size:12px;color:var(--muted);gap:4px 12px;max-width:62%;pointer-events:none}
.cui.br{bottom:14px;right:14px;font-size:12px;color:var(--faint);pointer-events:none;text-align:right}
.ib{width:32px;height:32px;display:inline-grid;place-items:center;border:1px solid var(--line);border-radius:8px;background:rgba(17,26,42,.85);cursor:pointer;color:var(--ink);font-size:15px;padding:0}
.ib[aria-pressed=true]{background:var(--ink);color:var(--bg)}
.lbls{position:absolute;inset:0;pointer-events:none;z-index:2;overflow:hidden}
.lbl{position:absolute;left:0;top:0;white-space:nowrap;font-size:11px;color:var(--muted);text-shadow:0 0 6px #0A101C,0 0 2px #0A101C;transition:opacity .4s;will-change:transform}
.lbl.g{color:#C9D3DF;font-weight:600;font-size:11.5px}
.lbl.r{color:var(--faint);font-size:10.5px}
.lbl.ax{color:var(--faint);font-size:10.5px}
.mark{position:absolute;left:0;top:0;width:28px;height:28px;margin:-14px 0 0 -14px;border:2px solid #fff;border-radius:50%;pointer-events:none;z-index:2;opacity:0;transition:opacity .3s}
.mark.on{opacity:1;animation:pulse 1.8s ease-out infinite}
@keyframes pulse{0%{box-shadow:0 0 0 0 rgba(255,255,255,.5)}100%{box-shadow:0 0 0 18px rgba(255,255,255,0)}}
#cloudMsg{position:absolute;inset:0;display:grid;place-items:center;color:var(--muted);font-size:14px;z-index:1;padding:20px;text-align:center}

/* mosaico */
#tree{position:relative;height:560px}
.tg{position:absolute;border-radius:7px;background:var(--panel);transition:left .7s var(--ease),top .7s var(--ease),width .7s var(--ease),height .7s var(--ease),opacity .4s;cursor:zoom-in;overflow:hidden}
.tg:hover{background:var(--panel2)}
.tg.focus{cursor:default;background:var(--panel)}
.tg .h{font-size:11.5px;font-weight:600;padding:3px 7px 0;white-space:nowrap;overflow:hidden;text-overflow:ellipsis;height:19px}
.tc{position:absolute;z-index:2;border-radius:2px;transition:left .7s var(--ease),top .7s var(--ease),width .7s var(--ease),height .7s var(--ease),opacity .4s;
  color:#0A101C;font-size:11px;line-height:1.25;overflow:hidden;padding:0;cursor:pointer}
.tc.lab{padding:4px 6px;font-weight:500}
.tc:hover{filter:brightness(1.25)}
.crumb{display:flex;gap:8px;align-items:center;font-size:13px;color:var(--muted);min-height:32px;flex-wrap:wrap}
.crumb button{background:none;border:0;color:var(--muted);cursor:pointer;padding:0;text-decoration:underline;text-underline-offset:3px}
.crumb b{color:var(--ink)}

/* ciudad 3D */
.citywrap{position:relative;height:clamp(500px,76vh,760px);border-radius:16px;overflow:hidden;border:1px solid var(--line);
  background:linear-gradient(180deg,#0B1322 0%,#111C30 55%,#0A101C 100%);touch-action:none}
#cityMsg{position:absolute;inset:0;display:grid;place-items:center;color:var(--muted);font-size:14px;z-index:1;padding:20px;text-align:center}
.segs{display:flex;gap:10px;flex-wrap:wrap}
.citywrap .lbl.g{font-size:11.5px;color:#DCE4EE}
.citywrap .lbl.g.on{color:#fff;text-decoration:underline;text-underline-offset:3px}
#tdrop>summary{list-style:none;cursor:pointer;display:inline-flex;align-items:center;gap:10px;padding:10px 16px;border:1px solid var(--line);border-radius:999px;background:var(--panel);font-weight:600;font-size:14px;margin-bottom:14px}
#tdrop>summary::-webkit-details-marker{display:none}
#tdrop>summary::before{content:'▸';transition:transform .25s var(--ease)}
#tdrop[open]>summary::before{transform:rotate(90deg)}
#tdrop>summary:hover{border-color:var(--muted)}
#tdrop .tdc,#tdrop[open] .tdo{display:none}
#tdrop[open] .tdc{display:inline}
.tsearch{display:flex;align-items:center;gap:14px;margin:0 0 12px;flex-wrap:wrap}
.tsearch input{max-width:420px;flex:1 1 280px}
mark{background:rgba(233,162,59,.35);color:var(--ink);border-radius:2px;padding:0 1px}

/* años */
#years{position:relative;height:320px;overflow:hidden}
#years.dropping .yd{transition:none}
#years .yc{transition:top .9s var(--ease),opacity .5s}
#years.dropping .yc{opacity:0}
.yd{position:absolute;left:0;top:0;border-radius:50%;transition:transform .9s var(--ease),opacity .45s;cursor:pointer}
.yd:hover{filter:brightness(1.35)}
.yl,.yc{position:absolute;text-align:center;font-size:12px;color:var(--muted);font-variant-numeric:tabular-nums;transition:top .9s var(--ease)}
.yc{color:var(--ink);font-weight:600}

/* ranking */
.rk-head,.rk{display:grid;grid-template-columns:minmax(0,1.25fr) minmax(0,2fr) minmax(130px,.9fr);gap:14px;align-items:center}
.rk-head{font-size:12px;color:var(--muted);padding:0 8px 8px;border-bottom:1px solid var(--line)}
.rk{width:100%;border:0;background:none;text-align:left;padding:8px;border-radius:8px;cursor:pointer;color:var(--ink)}
.rk:hover,.rk.sel{background:var(--panel)}
.rk .rn{font-size:13px;line-height:1.3;overflow:hidden;text-overflow:ellipsis;display:-webkit-box;-webkit-line-clamp:2;-webkit-box-orient:vertical}
.rk .rn small{color:var(--muted)}
.rk .rb{display:flex;height:16px;gap:2px;border-radius:4px;overflow:hidden}
.rk .rb span{min-width:0}
.rk .rm{display:flex;align-items:center;gap:8px;font-size:13px;font-variant-numeric:tabular-nums}
.rk .rm i{height:8px;border-radius:4px;background:var(--dead);opacity:.85;flex:none}

/* tabla */
.tablewrap{overflow-x:auto}
table{width:100%;border-collapse:collapse;font-size:13.5px}
th{text-align:left;font-weight:600;color:var(--muted);padding:0;border-bottom:1px solid var(--muted);white-space:nowrap}
th button{background:none;border:0;color:inherit;padding:8px 10px;cursor:pointer;font-weight:600}
th button[data-dir]::after{content:attr(data-dir);margin-left:4px;color:var(--ink)}
td{padding:9px 10px;border-bottom:1px solid var(--line);vertical-align:top}
tbody tr{cursor:pointer}
tbody tr:hover{background:var(--panel)}
.tag{display:inline-block;padding:2px 9px;border-radius:999px;color:#0A101C;font-size:12px;font-weight:700;white-space:nowrap}
.num{font-variant-numeric:tabular-nums}
.tools{display:flex;gap:10px;margin-top:14px;flex-wrap:wrap}
#metodo{margin-top:48px;border-top:1px solid var(--line);padding-top:16px;color:var(--muted);max-width:84ch}
#metodo summary{cursor:pointer;color:var(--ink);font-weight:600}
#metodo li{margin:7px 0}
#metodo b{color:var(--ink)}
footer{margin-top:28px;font-size:12.5px;color:var(--muted)}

/* tooltip y ficha */
#tip{position:fixed;left:0;top:0;z-index:60;pointer-events:none;max-width:300px;background:rgba(23,35,56,.97);border:1px solid #2C3D5A;border-radius:10px;
  padding:9px 11px;font-size:12.5px;line-height:1.4;box-shadow:0 10px 30px rgba(0,0,0,.45);opacity:0;transition:opacity .12s}
#tip .t{font-weight:600;color:var(--ink);margin-bottom:2px}
#tip .m{color:var(--muted)}
#scrim{position:fixed;inset:0;background:rgba(3,6,12,.5);opacity:0;pointer-events:none;transition:opacity .3s;z-index:49}
#scrim.open{opacity:1;pointer-events:auto}
@media (min-width:901px){#scrim{background:rgba(3,6,12,.25)}}
#ficha{position:fixed;top:0;right:0;height:100%;width:min(440px,100%);background:var(--panel);border-left:1px solid var(--line);
  transform:translateX(102%);transition:transform .4s var(--ease);z-index:50;overflow-y:auto;padding:26px 26px 40px;box-shadow:-20px 0 50px rgba(0,0,0,.35)}
#ficha.open{transform:none}
#ficha .x{position:absolute;top:14px;right:14px}
#ficha h3{font:600 23px/1.2 var(--serif);margin:12px 0 6px;padding-right:30px;text-wrap:balance}
#ficha .org{color:var(--muted);margin:0 0 18px;font-size:13.5px}
.big{display:flex;gap:12px;align-items:baseline;border-top:1px solid var(--line);border-bottom:1px solid var(--line);padding:14px 0;margin-bottom:14px}
.big b{font:700 40px/1 var(--serif);font-variant-numeric:tabular-nums}
.big span{color:var(--muted);font-size:13.5px;line-height:1.35}
.strip{width:100%;height:auto;display:block;margin:4px 0 2px}
dl.facts{display:grid;grid-template-columns:auto 1fr;gap:8px 14px;margin:18px 0;font-size:13.5px}
dl.facts dt{color:var(--muted)}
dl.facts dd{margin:0}
.meter{display:inline-block;width:80px;height:6px;border-radius:3px;background:var(--line);vertical-align:middle;margin-right:8px;overflow:hidden}
.meter i{display:block;height:100%;background:var(--ink)}
.acts{display:flex;flex-direction:column;gap:8px}
.acts .btn{justify-content:center}

@media (max-width:1100px){.kpis{grid-template-columns:repeat(2,minmax(0,1fr))}}
@media (max-width:900px){
  .shell{grid-template-columns:1fr;gap:20px;padding:36px 16px 48px}
  aside.side{position:static;max-height:none;background:var(--panel);border:1px solid var(--line);border-radius:12px;padding:14px}
  #filtros summary{margin:0}
  #filtros[open] summary{margin-bottom:14px}
  .step{align-items:flex-end;padding:0 16px 14vh}
  .step.title{align-items:center}
  .step .card{max-width:none}
  #dots{right:8px}
  .topbar{padding:14px 16px}
  .topbar .brand{display:none}
  .cui.bl{max-width:100%;bottom:44px}
  .cui.br{display:none}
  .rk-head,.rk{grid-template-columns:minmax(0,1fr) minmax(0,1.2fr);}
  .rk .rm{grid-column:2}
  .rk-head span:last-child{display:none}
  #tree{height:480px}
}
@media (prefers-reduced-motion:reduce){*{transition:none!important;animation:none!important}html{scroll-behavior:auto}}
</style>
</head>
<body>

<section id="story" aria-label="Introducción">
  <div id="stage">
    <div id="stage3d" class="gl"></div>
    <nav class="topbar"><span class="brand">El mapa de lo que no se mide</span><a href="#tablero">Saltar al tablero ↓</a></nav>
    <div id="dots" aria-hidden="true"></div>
  </div>
  <div id="steps"></div>
</section>

<div class="shell" id="tablero">
<aside class="side" aria-label="Filtros">
  <details id="filtros" open>
  <summary>Filtrar el catálogo</summary>
  <div class="fbody">
    <label>Buscar un conjunto u organismo
      <div class="sbox"><input id="q" type="search" autocomplete="off" placeholder="ej.: vacunas, INDEC, precios" role="combobox" aria-autocomplete="list" aria-expanded="false" aria-controls="sug"><ul id="sug" role="listbox" hidden></ul></div>
    </label>
    <label>Tema<select id="tema"></select></label>
    <label>Organismo<select id="org"></select></label>
    <label>Provincia mencionada<select id="prov"></select></label>
    <label><span>Sin actualizar hace al menos <b id="ageLbl">0 años</b></span><input id="age" type="range" min="0" max="9" step="0.5" value="0"></label>
    <div id="chips"></div>
    <button class="btn" id="reset" type="button">Limpiar filtros</button>
    <p class="small" id="count"></p>
    <p class="small">Los filtros quedan guardados en la dirección de la página: copiala para compartir esta vista.</p>
  </div>
  </details>
</aside>

<main>
  <p class="kicker">Explorá el catálogo · metadatos al <span class="gen"></span></p>
  <h1 id="hero"></h1>
  <div class="kpis">
    <div class="kpi"><b id="kMi">0</b><span>actualizaciones prometidas que nunca llegaron</span></div>
    <div class="kpi"><b id="kNoF">0</b><span id="kNoFl">no declaran cada cuánto se actualizan</span></div>
    <div class="kpi"><b id="kMed">0</b><span>años sin actualizar, en la mitad de los conjuntos (mediana)</span></div>
    <div class="kpi"><b id="kProv">0</b><span id="kProvl">dicen de qué provincia hablan</span></div>
  </div>
  <div id="bar" role="group" aria-label="Estado de los conjuntos de datos"></div>
  <div id="legend"></div>

  <section class="blk" id="nube">
    <h2>La nube del olvido</h2>
    <p class="lede">Cada punto es un conjunto de datos y su color, el estado. Cada porción del disco es un <b>organismo</b> (o un tema). La <b>distancia al centro</b> son los años sin actualizar. La <b>altura</b> son las actualizaciones prometidas que nunca llegaron. Arrastrá para girar, usá la rueda o pellizcá para acercarte y tocá un punto para abrir su ficha. Lo que queda fuera de los filtros pasa a la sombra.</p>
    <div class="cloudwrap">
      <div id="cloud3d" class="gl"></div>
      <div id="cloudMsg">Cargando la vista 3D…</div>
      <div class="cui tl"><div class="seg segG" role="group" aria-label="Agrupar por"><button type="button" data-g="p">Organismo</button><button type="button" data-g="th">Tema</button></div></div>
      <div class="cui tr" role="group" aria-label="Controles de cámara">
        <button class="ib" type="button" data-view="3d" title="Vista en perspectiva" aria-label="Vista en perspectiva">◇</button>
        <button class="ib" type="button" data-view="top" title="Vista desde arriba: organismo y antigüedad" aria-label="Vista desde arriba">◎</button>
        <button class="ib" type="button" data-view="side" title="Vista de costado: antigüedad y actualizaciones que faltan" aria-label="Vista de costado">▭</button>
        <button class="ib" type="button" id="zIn" title="Acercar" aria-label="Acercar">+</button>
        <button class="ib" type="button" id="zOut" title="Alejar" aria-label="Alejar">−</button>
        <button class="ib" type="button" id="spin" title="Rotación automática" aria-label="Rotación automática" aria-pressed="true">⟳</button>
      </div>
      <div class="cui bl" id="cloudLegend"></div>
      <div class="cui br">Arrastrá para girar · clic sobre un punto para ver su ficha</div>
    </div>
    <p class="small" id="cloudNote"></p>
  </section>

  <section class="blk" id="mosaico">
    <h2 id="mosTitle">La ciudad del olvido</h2>
    <p class="lede" id="mosLede"></p>
    <div class="sechead"><div class="crumb" id="crumb"></div>
      <div class="segs"><div class="seg" role="group" aria-label="Tipo de vista"><button type="button" data-mv="3d">Ciudad 3D</button><button type="button" data-mv="2d">Mosaico 2D</button></div>
      <div class="seg segG" role="group" aria-label="Agrupar por"><button type="button" data-g="p">Organismo</button><button type="button" data-g="th">Tema</button></div></div></div>
    <div class="citywrap" id="city">
      <div id="city3d" class="gl"></div>
      <div id="cityMsg">Cargando la ciudad 3D…</div>
      <div class="cui tr" role="group" aria-label="Controles de cámara de la ciudad">
        <button class="ib" type="button" id="cHome" title="Vista inicial" aria-label="Vista inicial">⌂</button>
        <button class="ib" type="button" id="cIn" title="Acercar" aria-label="Acercar">+</button>
        <button class="ib" type="button" id="cOut" title="Alejar" aria-label="Alejar">−</button>
        <button class="ib" type="button" id="cSpin" title="Rotación automática" aria-label="Rotación automática" aria-pressed="true">⟳</button>
      </div>
      <div class="cui bl" id="cityLegend"></div>
      <div class="cui br">Arrastrá para girar · clic en un edificio para ver su ficha · clic en una manzana para filtrar</div>
    </div>
    <div id="tree" hidden></div>
  </section>

  <section class="blk" id="anios">
    <div class="sechead"><div><h2>El año en que dejaron de actualizarse</h2>
    <p class="lede">Cada punto es un conjunto, apilado en el año de su última actualización. Las columnas de años viejos son datos que se publicaron y nadie volvió a tocar. <span id="yearsNote"></span></p></div>
    <button class="btn" id="yReplay" type="button">↻ Repetir la caída</button></div>
    <div id="years"></div>
  </section>

  <section class="blk" id="ranking">
    <h2>Qué organismos cumplen menos lo que prometen</h2>
    <div class="sechead"><p class="lede">Los 15 organismos con 5 o más conjuntos que peor cumplen. Tocá uno para filtrar todo el tablero.</p>
      <div class="seg" role="group" aria-label="Ordenar ranking"><button type="button" id="rkPct" aria-pressed="true">% que incumple</button><button type="button" id="rkMi" aria-pressed="false">Actualizaciones que faltan</button></div></div>
    <div class="rk-head"><span>Organismo (conjuntos)</span><span>Composición por estado</span><span>Actualizaciones que faltan</span></div>
    <div id="rank"></div>
  </section>

  <section class="blk" id="tabla">
    <h2>Los conjuntos más viejos</h2>
    <p class="lede">Ordenados por años sin actualizar. Buscá dentro de la tabla, tocá una columna para reordenar y una fila para abrir la ficha.</p>
    <details id="tdrop">
    <summary><span class="tdo">Ver la tabla</span><span class="tdc">Ocultar la tabla</span> <span class="small" id="tsum"></span></summary>
    <div class="tsearch"><input id="tq" type="search" autocomplete="off" placeholder="Buscar en la tabla: título, organismo, tema o frecuencia" aria-label="Buscar en la tabla"><span class="small" id="tcount"></span></div>
    <div class="tablewrap"><table><thead><tr id="thead"></tr></thead><tbody id="rows"></tbody></table></div>
    <div class="tools"><button class="btn" id="more" type="button">Mostrar más</button><button class="btn" id="csv" type="button">Descargar CSV filtrado</button></div>
    </details>
  </section>

  <details id="metodo">
    <summary>Cómo se calculó</summary>
    <ul>
      <li><b>Fuente:</b> la API CKAN del portal Datos Argentina (datos.gob.ar/api/3/action/package_search), consultada el <span class="gen"></span>. Todo el análisis usa los metadatos que publica cada organismo, no el contenido de los archivos.</li>
      <li><b>Última actualización:</b> la fecha más reciente entre la del conjunto y las de sus recursos. Es la fecha que declara el catálogo, así que puede reflejar un cambio en la ficha y no en los datos. No se usa <i>metadata_modified</i>, porque en el portal nuevo es la fecha de la migración.</li>
      <li><b>Estado:</b> la antigüedad se compara con la frecuencia que declaró el propio organismo. <i>Al día</i>: hasta 1,5 períodos (mínimo 14 días). <i>Atrasado</i>: hasta 4 períodos (mínimo 90 días). <i>Abandonado</i>: más que eso. Si no declara frecuencia, cuenta como abandonado después de 3 años sin cambios.</li>
      <li><b>Actualizaciones que faltan:</b> cuántas veces debió actualizarse según su frecuencia desde la última actualización (antigüedad ÷ período, sin decimales). Se cuenta como máximo una por día, aunque la ficha prometa actualizaciones cada hora o cada segundo. Vale 0 para los conjuntos al día y para los que no declaran frecuencia.</li>
      <li><b>Nube 3D:</b> el ángulo agrupa por organismo o tema, la distancia al centro es lineal en años y la altura usa escala logarítmica (log<sub>10</sub> de 1 + actualizaciones que faltan). La posición dentro de cada porción es un desplazamiento fijo para que los puntos no se superpongan, y no codifica ningún dato.</li>
      <li><b>Ciudad 3D:</b> cada manzana agrupa los conjuntos de un organismo o tema, ordenadas de mayor a menor cantidad. La altura de cada edificio es lineal en años sin actualizar y el color, su estado. La proporción de ventanas encendidas repite el estado como textura (al día, casi todas; abandonado, casi ninguna) y no agrega otro dato. Dentro de cada manzana, los edificios más altos van al fondo.</li>
      <li><b>Provincias:</b> un conjunto cuenta para una provincia si la nombra en el título, la descripción, las palabras clave o la cobertura espacial, o si lo dice el campo de provincia del portal. Es una detección por texto y puede fallar con nombres ambiguos.</li>
      <li><b>Metadatos:</b> puntaje de 0 a 100 según si el conjunto tiene descripción, tema, palabras clave, frecuencia, cobertura, licencia y enlace de descarga.</li>
    </ul>
  </details>
  <footer>Datos: Datos Argentina (datos.gob.ar). Análisis y visualización para el concurso Contar con Datos (UdeSA).</footer>
</main>
</div>

<div id="tip" role="tooltip"></div>
<div id="scrim"></div>
<aside id="ficha" role="dialog" aria-modal="true" aria-label="Ficha del conjunto de datos" aria-hidden="true"></aside>

<script id="data" type="application/json">__DATA__</script>
<script>
(function(){
const raw = JSON.parse(document.getElementById('data').textContent);
const DATA = raw.datasets, META = raw.meta, PROV = raw.provincias || [];
const ESTADOS = ['Al día','Atrasado','Abandonado','Sin periodicidad','Sin fecha'];
const COLOR = {'Al día':'#35B8A8','Atrasado':'#E9A23B','Abandonado':'#E0507A','Sin periodicidad':'#9AA6B8','Sin fecha':'#4E5A6C'};
const ORD = {}; ESTADOS.forEach((e,i)=>ORD[e]=i);
const REDUCED = matchMedia('(prefers-reduced-motion: reduce)').matches;
const N = DATA.length;
const norm = s => (s||'').normalize('NFD').replace(/[̀-ͯ]/g,'').toLowerCase();
const $ = id => document.getElementById(id);
const esc = s => String(s==null?'':s).replace(/[&<>"']/g, c => ({'&':'&amp;','<':'&lt;','>':'&gt;','"':'&quot;',"'":'&#39;'}[c]));
const fmt = n => Math.round(n).toLocaleString('es-AR');
const fmt1 = n => (Math.round(n*10)/10).toLocaleString('es-AR');
const pct = (a,b) => b ? Math.round(100*a/b) : 0;
const fmtAge = a => a==null ? 's/d' : (a<1 ? 'menos de 1 año' : fmt1(a)+' años');
const fdate = d => d ? d.slice(8,10)+'/'+d.slice(5,7)+'/'+d.slice(0,4) : 's/d';
const isBad = r => r.st==='Atrasado' || r.st==='Abandonado';
const genDate = (()=>{ const m=(META.generado||'').split('/'); return m.length===3 ? new Date(+m[2], +m[1]-1, +m[0]) : new Date(); })();
DATA.forEach(r => { r._s = norm(r.t+' '+r.p+' '+r.th); r.mi = r.mi||0; r.pv = r.pv||[]; });
const BY_I = new Map(DATA.map(r=>[r.i,r]));
document.querySelectorAll('.gen').forEach(e=>e.textContent=META.generado);

const APP = window.APP = {DATA, COLOR, ESTADOS, REDUCED, N};

/* ── estado + dirección compartible ── */
const st = {tema:'', org:'', prov:'', q:'', minAge:0, estados:new Set(ESTADOS), group:'p', shown:60, sort:{k:'age',dir:-1}, rankBy:'pct', focus:null, mv:'3d', tq:''};
APP.st = st;
function readHash(){
  let h; try{ h = new URLSearchParams(location.hash.slice(1)); }catch(e){ return; }
  const has = (k,v) => DATA.some(r=>r[k]===v);
  if(h.get('tema') && has('th',h.get('tema'))) st.tema = h.get('tema');
  if(h.get('org') && has('p',h.get('org'))) st.org = h.get('org');
  if(h.get('prov') && DATA.some(r=>r.pv.includes(h.get('prov')))) st.prov = h.get('prov');
  if(h.get('q')) st.q = h.get('q');
  if(h.get('edad')) st.minAge = Math.max(0, Math.min(9, +h.get('edad')||0));
  if(h.get('estados')){ const s = h.get('estados').split('|').filter(e=>ORD[e]!=null); if(s.length) st.estados = new Set(s); }
  if(h.get('g')==='th') st.group = 'th';
  if(h.get('vista')==='2d') st.mv = '2d';
}
function writeHash(){
  const h = new URLSearchParams();
  if(st.tema) h.set('tema',st.tema); if(st.org) h.set('org',st.org); if(st.prov) h.set('prov',st.prov);
  if(st.q) h.set('q',st.q); if(st.minAge) h.set('edad',st.minAge);
  if(st.estados.size<ESTADOS.length) h.set('estados',[...st.estados].join('|'));
  if(st.group==='th') h.set('g','th');
  if(st.mv==='2d') h.set('vista','2d');
  const s = h.toString();
  try{ history.replaceState(null,'', s ? '#'+s : location.pathname+location.search); }catch(e){}
}

function rows(ignoreEstado){
  const q = norm(st.q.trim());
  return DATA.filter(r =>
    (!st.tema || r.th===st.tema) && (!st.org || r.p===st.org) && (!st.prov || r.pv.includes(st.prov)) &&
    (ignoreEstado || st.estados.has(r.st)) &&
    (st.minAge===0 || (r.age!=null && r.age>=st.minAge)) && (!q || r._s.includes(q)));
}
APP.rows = rows;
function counts(arr){ const c={}; ESTADOS.forEach(e=>c[e]=0); arr.forEach(r=>c[r.st]++); return c; }
function median(a){ if(!a.length) return 0; const s=a.slice().sort((x,y)=>x-y), m=s.length>>1; return s.length%2 ? s[m] : (s[m-1]+s[m])/2; }

/* ── animaciones de números ── */
function tween(el, to, f){
  const from = el._v==null ? 0 : el._v; el._v = to;
  cancelAnimationFrame(el._raf);
  if(REDUCED || from===to){ el.textContent = f(to); return; }
  const t0 = performance.now(), D = 1000;
  const step = now => { const k=Math.min(1,(now-t0)/D), e=1-Math.pow(1-k,3); el.textContent = f(from+(to-from)*e); if(k<1) el._raf=requestAnimationFrame(step); };
  el._raf = requestAnimationFrame(step);
}

/* ── tooltip ── */
const tip = $('tip');
APP.tip = {
  show(html, x, y){
    tip.innerHTML = html; tip.style.opacity = 1;
    const w = tip.offsetWidth, h = tip.offsetHeight;
    let L = x+16, T = y+16;
    if(L+w > innerWidth-8) L = x-w-16; if(T+h > innerHeight-8) T = y-h-16;
    tip.style.transform = 'translate('+Math.max(8,L)+'px,'+Math.max(8,T)+'px)';
  },
  hide(){ tip.style.opacity = 0; }
};
function tipHTML(r){
  return '<div class="t">'+esc(r.t)+'</div><div class="m">'+esc(r.p)+'</div>'+
    '<div><span class="dot" style="background:'+COLOR[r.st]+'"></span>'+r.st+' · última act. '+fdate(r.d)+'</div>'+
    '<div class="m">'+(r.pr ? 'Prometía: '+esc(r.f).toLowerCase()+(r.mi ? ' · faltan <b style="color:#E7EDF4">'+fmt(r.mi)+'</b> actualizaciones' : '') : 'No declara frecuencia')+'</div>';
}
APP.tipHTML = tipHTML;

/* ── filtros ── */
function fillSelect(el, getter, label){
  const m = new Map(); DATA.forEach(r => [].concat(getter(r)).forEach(k => m.set(k,(m.get(k)||0)+1)));
  const opts = [...m].sort((a,b)=>a[0].localeCompare(b[0],'es'));
  el.innerHTML = '<option value="">'+label+'</option>' + opts.map(([k,n])=>'<option value="'+esc(k)+'">'+esc(k)+' ('+n+')</option>').join('');
}
fillSelect($('tema'), r=>r.th, 'Todos los temas');
fillSelect($('org'), r=>r.p, 'Todos los organismos');
fillSelect($('prov'), r=>r.pv, 'Cualquiera (o ninguna)');
const nProv = DATA.filter(r=>r.pv.length).length;
$('prov').insertAdjacentHTML('afterend','<span class="small">Solo '+nProv+' de '+fmt(N)+' conjuntos nombran alguna provincia.</span>');

function syncControls(){
  $('tema').value = st.tema; $('org').value = st.org; $('prov').value = st.prov; $('q').value = st.q; $('age').value = st.minAge;
  $('ageLbl').textContent = st.minAge===0 ? '0 años' : fmt1(st.minAge)+' años';
  document.querySelectorAll('.segG button').forEach(b => b.setAttribute('aria-pressed', b.dataset.g===st.group));
  document.querySelectorAll('[data-mv]').forEach(b => b.setAttribute('aria-pressed', b.dataset.mv===st.mv));
}
const changed = () => { st.shown = 60; render(); };
$('tema').onchange = e => { st.tema = e.target.value; changed(); };
$('org').onchange  = e => { st.org  = e.target.value; changed(); };
$('prov').onchange = e => { st.prov = e.target.value; changed(); };
$('age').oninput   = e => { st.minAge = +e.target.value; syncControls(); changed(); };
document.querySelectorAll('.segG button').forEach(b => b.onclick = () => { st.group = b.dataset.g; st.focus = null; syncControls(); render(); });
$('reset').onclick = () => {
  Object.assign(st,{tema:'',org:'',prov:'',q:'',minAge:0,estados:new Set(ESTADOS),shown:60,focus:null});
  syncControls(); closeSug(); render();
};

/* buscador con sugerencias */
const qEl = $('q'), sug = $('sug'); let sugList = [], sugIdx = -1;
function closeSug(){ sug.hidden = true; qEl.setAttribute('aria-expanded','false'); sugIdx = -1; }
function renderSug(){
  const q = norm(qEl.value.trim());
  if(q.length < 2){ closeSug(); return; }
  sugList = DATA.filter(r => r._s.includes(q))
    .sort((a,b) => (norm(a.t).includes(q)?0:1)-(norm(b.t).includes(q)?0:1) || b.mi-a.mi).slice(0,7);
  if(!sugList.length){ closeSug(); return; }
  sug.innerHTML = sugList.map((r,k) => '<li role="option" id="sg'+k+'" data-i="'+r.i+'" aria-selected="'+(k===sugIdx)+'"><span class="dot" style="background:'+COLOR[r.st]+'"></span>'+esc(r.t)+'<small>'+esc(r.p)+'</small></li>').join('');
  sug.hidden = false; qEl.setAttribute('aria-expanded','true');
}
qEl.oninput = () => { st.q = qEl.value; sugIdx = -1; renderSug(); changed(); };
qEl.onkeydown = e => {
  if(sug.hidden) return;
  if(e.key==='ArrowDown'||e.key==='ArrowUp'){ e.preventDefault(); sugIdx = (sugIdx + (e.key==='ArrowDown'?1:-1) + sugList.length) % sugList.length; renderSug(); qEl.setAttribute('aria-activedescendant','sg'+sugIdx); }
  else if(e.key==='Enter'){ if(sugIdx>=0){ e.preventDefault(); pick(sugList[sugIdx].i); } else closeSug(); }
  else if(e.key==='Escape') closeSug();
};
qEl.onblur = () => setTimeout(closeSug, 150);
sug.onmousedown = e => { const li = e.target.closest('li'); if(li){ e.preventDefault(); pick(+li.dataset.i); } };
function pick(i){ closeSug(); locate(i); openFicha(i); }
function locate(i){
  $('nube').scrollIntoView({behavior: REDUCED?'auto':'smooth', block:'start'});
  if(APP.cloud) APP.cloud.focus(i);
}

/* ── titular, indicadores y barra ── */
function renderHero(base){
  const prom = base.filter(r=>r.pr===1), inc = prom.filter(isBad).length;
  const h = $('hero');
  if(prom.length){
    if(!$('heroN')) h.innerHTML = '<span class="num" id="heroN">0</span> de cada 100 conjuntos de datos que prometen actualizarse ya no lo hacen.';
    tween($('heroN'), pct(inc,prom.length), v=>fmt(v));
  } else h.textContent = 'Ningún conjunto con estos filtros declara cada cuánto se actualiza.';
  tween($('kMi'), base.reduce((s,r)=>s+r.mi,0), fmt);
  const nf = base.filter(r=>r.pr===0).length;
  tween($('kNoF'), nf, fmt); $('kNoFl').textContent = 'no declaran cada cuánto se actualizan ('+pct(nf,base.length)+' %)';
  tween($('kMed'), median(base.filter(r=>r.age!=null).map(r=>r.age)), fmt1);
  const np = base.filter(r=>r.pv.length).length;
  tween($('kProv'), pct(np,base.length), v=>fmt(v)+' %'); $('kProvl').textContent = 'dicen de qué provincia hablan ('+fmt(np)+' de '+fmt(base.length)+')';
}
function renderBar(base, arr){
  const c = counts(base), tot = base.length || 1;
  $('bar').innerHTML = ESTADOS.filter(e=>c[e]>0).map(e =>
    '<button type="button" data-e="'+e+'" class="'+(st.estados.has(e)?'':'off')+'" style="flex:'+c[e]+' 1 0;background:'+COLOR[e]+'" title="'+e+': '+c[e]+'" aria-pressed="'+st.estados.has(e)+'">'+(c[e]/tot>.07 ? pct(c[e],tot)+' %' : '')+'</button>').join('');
  $('bar').querySelectorAll('button').forEach(b => b.onclick = () => {
    const e = b.dataset.e;
    st.estados = (st.estados.size===1 && st.estados.has(e)) ? new Set(ESTADOS) : new Set([e]);
    changed();
  });
  $('legend').innerHTML = ESTADOS.filter(e=>c[e]>0).map(e => '<span><span class="dot" style="background:'+COLOR[e]+'"></span>'+e+' · '+fmt(c[e])+'</span>').join('') + '<span class="small">Tocá un color para aislarlo y tocalo otra vez para volver.</span>';
  $('count').textContent = fmt(arr.length)+' de '+fmt(N)+' conjuntos coinciden con los filtros.';
  const chips = [];
  if(st.org) chips.push(['org',st.org]); if(st.tema) chips.push(['tema',st.tema]); if(st.prov) chips.push(['prov',st.prov]);
  $('chips').innerHTML = chips.map(([k,v]) => '<button class="chip" type="button" data-k="'+k+'">'+esc(v)+' <b aria-label="Quitar filtro">×</b></button>').join(' ');
  $('chips').querySelectorAll('button').forEach(b => b.onclick = () => { st[b.dataset.k]=''; syncControls(); changed(); });
}

/* ── mosaico (treemap propio con zoom) ── */
const treeEl = $('tree'); const tG = new Map(), tC = new Map();
function squarify(items, x, y, w, h){
  const total = items.reduce((s,a)=>s+a.v,0); if(!total || w<=0 || h<=0) return [];
  const scale = w*h/total, out = [];
  let rest = items.map(a=>({a, area:a.v*scale})), rx=x, ry=y, rw=w, rh=h;
  const worst = (row, side) => { const s=row.reduce((t,c)=>t+c.area,0); let mx=0, mn=Infinity; row.forEach(c=>{mx=Math.max(mx,c.area); mn=Math.min(mn,c.area);}); return Math.max(side*side*mx/(s*s), s*s/(side*side*mn)); };
  while(rest.length){
    const side = Math.min(rw, rh); let row = [rest[0]], i = 1;
    while(i<rest.length && worst(row.concat(rest[i]),side) <= worst(row,side)){ row.push(rest[i]); i++; }
    rest = rest.slice(row.length);
    const s = row.reduce((t,c)=>t+c.area,0);
    if(rw >= rh){ const cw = s/rh; let cy = ry; row.forEach(c=>{ const ch=c.area/cw; out.push({it:c.a,x:rx,y:cy,w:cw,h:ch}); cy+=ch; }); rx+=cw; rw-=cw; }
    else { const ch = s/rw; let cx = rx; row.forEach(c=>{ const cw=c.area/ch; out.push({it:c.a,x:cx,y:ry,w:cw,h:ch}); cx+=cw; }); ry+=ch; rh-=ch; }
  }
  return out;
}
function cellEl(r){
  let el = tC.get(r.i);
  if(!el){ el = document.createElement('div'); el.className='tc'; el.dataset.i=r.i; el.style.background=COLOR[r.st]; el.style.opacity=0; treeEl.appendChild(el); tC.set(r.i,el); }
  return el;
}
function groupStats(a){ const ab=a.filter(r=>r.st==='Abandonado').length, bad=a.filter(isBad).length; return {n:a.length, ab, bad, mi:a.reduce((s,r)=>s+r.mi,0)}; }
function renderTree(arr){
  if(treeEl.hidden) return;
  const W = treeEl.clientWidth, H = treeEl.clientHeight, key = st.group;
  const groups = new Map(); arr.forEach(r => { const g=r[key]; if(!groups.has(g)) groups.set(g,[]); groups.get(g).push(r); });
  if(st.focus && !groups.has(st.focus)) st.focus = null;
  let items = [...groups].map(([g,a])=>({g,a,v:a.length})).sort((x,y)=>y.v-x.v);
  if(st.focus) items = items.filter(x=>x.g===st.focus);
  const word = key==='p' ? 'organismos' : 'temas';
  $('crumb').innerHTML = st.focus ? '<button type="button" id="crumbAll">Todos los '+word+'</button> › <b>'+esc(st.focus)+'</b> <span>· '+groups.get(st.focus).length+' conjuntos</span>'
    : '<span>'+groups.size+' '+word+' · '+fmt(arr.length)+' conjuntos</span>';
  if(st.focus) $('crumbAll').onclick = () => { st.focus=null; renderTree(rows()); };
  const seenG = new Set(), seenC = new Set();
  squarify(items, 0, 0, W, H).forEach(({it,x,y,w,h}) => {
    const id = key+'|'+it.g; seenG.add(id);
    let el = tG.get(id);
    if(!el){ el=document.createElement('div'); el.className='tg'; el.innerHTML='<div class="h"></div>'; el.dataset.g=it.g;
      el.style.cssText='left:'+(x+w/2)+'px;top:'+(y+h/2)+'px;width:0;height:0;opacity:0'; treeEl.insertBefore(el, treeEl.firstChild); tG.set(id,el); void el.offsetWidth; }
    el.style.left=(x+1)+'px'; el.style.top=(y+1)+'px'; el.style.width=Math.max(0,w-2)+'px'; el.style.height=Math.max(0,h-2)+'px'; el.style.opacity=1; el.style.pointerEvents='auto';
    el.classList.toggle('focus', !!st.focus);
    const showH = !st.focus && w>70 && h>40;
    const s = groupStats(it.a);
    el.querySelector('.h').textContent = showH ? it.g+' · '+it.v : '';
    el._info = '<div class="t">'+esc(it.g)+'</div><div>'+s.n+' conjuntos · '+pct(s.ab,s.n)+' % abandonados</div><div class="m">Faltan '+fmt(s.mi)+' actualizaciones prometidas'+(st.focus?'':' · clic para ampliar')+'</div>';
    const hdr = showH ? 20 : 3;
    const cx = x+3, cy = y+hdr+1, cw = Math.max(1,w-6), ch = Math.max(1,h-hdr-4);
    const sorted = it.a.slice().sort((p,q)=>ORD[p.st]-ORD[q.st] || (q.age||0)-(p.age||0));
    const n = sorted.length;
    let cols = Math.max(1, Math.round(Math.sqrt(n*cw/ch))); const rws = Math.ceil(n/cols); cols = Math.ceil(n/rws);
    const sw = cw/cols, sh = ch/rws, gap = st.focus ? 2 : 1;
    const lab = !!st.focus && sw>96 && sh>34;
    sorted.forEach((r,j) => {
      const c = cellEl(r); seenC.add(r.i);
      c.style.left=(cx+(j%cols)*sw)+'px'; c.style.top=(cy+Math.floor(j/cols)*sh)+'px';
      c.style.width=Math.max(1,sw-gap)+'px'; c.style.height=Math.max(1,sh-gap)+'px';
      c.style.opacity=1; c.style.pointerEvents = st.focus ? 'auto' : 'none';
      const txt = lab ? r.t : '';
      if(c._txt !== txt){ c.textContent = txt; c._txt = txt; c.classList.toggle('lab', lab); }
    });
  });
  tG.forEach((el,id) => { if(!seenG.has(id)){ el.style.opacity=0; el.style.pointerEvents='none'; } });
  tC.forEach((el,i) => { if(!seenC.has(i)){ el.style.opacity=0; el.style.pointerEvents='none'; } });
}
treeEl.addEventListener('mousemove', e => {
  const c = e.target.closest('.tc'), g = e.target.closest('.tg');
  if(c && c.style.pointerEvents!=='none') APP.tip.show(tipHTML(BY_I.get(+c.dataset.i)), e.clientX, e.clientY);
  else if(g && g._info) APP.tip.show(g._info, e.clientX, e.clientY);
  else APP.tip.hide();
});
treeEl.addEventListener('mouseleave', () => APP.tip.hide());
treeEl.addEventListener('click', e => {
  const c = e.target.closest('.tc'), g = e.target.closest('.tg');
  if(c && st.focus){ openFicha(+c.dataset.i); return; }
  if(g && !st.focus){ st.focus = g.dataset.g; APP.tip.hide(); renderTree(rows()); }
});

/* ── ciudad 3D / mosaico 2D ── */
const LEDE = {
  '3d': 'Cada <b>manzana</b> es un organismo (o un tema) y cada <b>edificio</b>, un conjunto de datos. La <b>altura</b> son los años que lleva sin actualizarse y el color, su estado. Las ventanas acompañan el estado: cuanto más abandonado está un conjunto, menos luces quedan prendidas. Las torres más altas son los datos más olvidados. Tocá un edificio para ver su ficha o la base de una manzana para filtrar por ese grupo.',
  '2d': 'Cada rectángulo grande es un organismo (o un tema), y su tamaño, la cantidad de conjuntos que publica. Adentro, cada celda es un conjunto pintado según su estado. Hacé clic en un grupo para ampliarlo y en una celda para ver su ficha.'
};
function setMosView(v){
  st.mv = v; syncControls();
  $('mosTitle').textContent = v==='3d' ? 'La ciudad del olvido' : 'Dónde se concentra el olvido';
  $('mosLede').innerHTML = LEDE[v];
  $('city').hidden = v!=='3d'; treeEl.hidden = v!=='2d';
  renderMos(rows()); writeHash();
}
document.querySelectorAll('[data-mv]').forEach(b => b.onclick = () => setMosView(b.dataset.mv));
function renderMos(arr){
  if(st.mv==='2d') renderTree(arr);
  else {
    const n = new Set(arr.map(r=>r[st.group])).size;
    $('crumb').innerHTML = '<span>'+n+' '+(st.group==='p'?'manzanas (organismos)':'manzanas (temas)')+' · '+fmt(arr.length)+' edificios</span>';
    if(APP.city) APP.city.update(arr, st.group);
  }
}
APP.filterBlock = name => { const k = st.group==='p' ? 'org' : 'tema'; st[k] = st[k]===name ? '' : name; syncControls(); changed(); };
APP.blockSel = () => st.group==='p' ? st.org : st.tema;
APP.cityReady = () => { const m=$('cityMsg'); if(m) m.remove(); renderMos(rows()); };
APP.cityFail = () => { const m=$('cityMsg'); if(m) m.innerHTML = 'La ciudad 3D necesita WebGL y conexión a internet. <button class="btn" type="button" id="to2d">Ver el mosaico 2D</button>'; const b=$('to2d'); if(b) b.onclick=()=>setMosView('2d'); };
function cityLegend(base){
  const c = counts(base);
  $('cityLegend').innerHTML = ESTADOS.filter(e=>c[e]>0).map(e=>'<span><span class="dot" style="background:'+COLOR[e]+'"></span>'+e+'</span>').join('');
}

/* ── años: un punto por conjunto ── */
const yEl = $('years'); const yDots = new Map(); let yearsSeen = false;
const YRS = (()=>{ const ys = DATA.filter(r=>r.d).map(r=>+r.d.slice(0,4)); const a=[]; if(!ys.length) return a; for(let y=Math.min(...ys); y<=Math.max(...ys); y++) a.push(y); return a; })();
const YCOUNT = new Map(YRS.map(y=>[y,0])); DATA.forEach(r=>{ if(r.d) YCOUNT.set(+r.d.slice(0,4), YCOUNT.get(+r.d.slice(0,4))+1); });
const YMAX = Math.max(1, ...YCOUNT.values());
const yLbl = document.createElement('div'); yEl.appendChild(yLbl);
function renderYears(arr){
  const W = yEl.clientWidth, n = YRS.length || 1, colW = W/n;
  const step = W < 640 ? 5 : (W < 900 ? 7 : 8), d = Math.max(3, step-1.5);
  const k = Math.max(2, Math.floor((colW-8)/step));
  const rowsMax = Math.ceil(YMAX/k), top = 26, H = top + rowsMax*step + 30;
  yEl.style.height = H+'px';
  const by = new Map(YRS.map(y=>[y,[]])); arr.forEach(r => { if(r.d && by.has(+r.d.slice(0,4))) by.get(+r.d.slice(0,4)).push(r); });
  const vis = new Set(), drops = []; let lab = '';
  YRS.forEach((y,yi) => {
    const a = by.get(y).sort((p,q)=>ORD[p.st]-ORD[q.st] || (q.mi-p.mi));
    const x0 = yi*colW + (colW - k*step)/2;
    a.forEach((r,j) => {
      vis.add(r.i);
      let el = yDots.get(r.i);
      if(!el){ el = document.createElement('div'); el.className='yd'; el.dataset.i=r.i; el.style.background=COLOR[r.st]; el.style.opacity=0;
        el.style.transform='translate('+(x0+(j%k)*step)+'px,-20px)'; yEl.appendChild(el); yDots.set(r.i,el); }
      el.style.width = el.style.height = d+'px';
      if(yearsSeen){
        const tx = x0+(j%k)*step, ty = top+(rowsMax-1-Math.floor(j/k))*step;
        el.style.transform = 'translate('+tx+'px,'+ty+'px)';
        el.style.opacity = 1; el.style.pointerEvents = 'auto';
        if(dropNext) drops.push({el, tx, ty, yi, row:Math.floor(j/k), i:r.i});
      }
    });
    const used = Math.ceil(a.length/k);
    lab += '<div class="yl" style="left:'+(yi*colW)+'px;width:'+colW+'px;top:'+(top+rowsMax*step+8)+'px">'+(W<640 ? "'"+String(y).slice(2) : y)+'</div>';
    if(a.length) lab += '<div class="yc" style="left:'+(yi*colW)+'px;width:'+colW+'px;top:'+(top+(rowsMax-used)*step-20)+'px">'+a.length+'</div>';
  });
  yLbl.innerHTML = lab;
  yDots.forEach((el,i) => { if(!vis.has(i)){ el.style.opacity=0; el.style.pointerEvents='none'; } });
  if(dropNext){ dropNext = false; drop(drops); }
  const lastY = YRS[YRS.length-1], old = YRS.filter(y=>y<=lastY-3).map(y=>[y,by.get(y).length]).sort((a,b)=>b[1]-a[1])[0];
  const stale = arr.filter(r=>r.d && +r.d.slice(0,4)<=lastY-3).length;
  $('yearsNote').innerHTML = old && old[1] ? 'Con estos filtros, <b>'+fmt(stale)+'</b> conjuntos no cambian desde '+(lastY-3)+' o antes; el año más congelado es <b>'+old[0]+'</b>, con '+fmt(old[1])+'.' : '';
}
/* caída con rebote: columna por columna, de abajo hacia arriba */
let dropNext = false, dropTimer = 0;
const jit = i => { const x = Math.sin(i*91.7+3.1)*43758.5453; return x-Math.floor(x); };
function drop(list){
  if(REDUCED || !list.length || !yEl.animate) return;
  clearTimeout(dropTimer); yEl.classList.add('dropping');
  let end = 0;
  list.forEach(o => {
    const delay = o.yi*110 + o.row*9 + jit(o.i)*160, from = -12 - jit(o.i*3)*150;
    end = Math.max(end, delay+760);
    o.el.getAnimations().forEach(a=>a.cancel());
    o.el.animate([
      {transform:'translate('+o.tx+'px,'+from+'px)', opacity:1, easing:'cubic-bezier(.55,0,1,.45)'},
      {transform:'translate('+o.tx+'px,'+o.ty+'px)', offset:.78, easing:'cubic-bezier(0,.55,.45,1)'},
      {transform:'translate('+o.tx+'px,'+(o.ty-5)+'px)', offset:.89, easing:'cubic-bezier(.55,0,1,.45)'},
      {transform:'translate('+o.tx+'px,'+o.ty+'px)', opacity:1}
    ], {duration:760, delay, fill:'backwards'});
  });
  dropTimer = setTimeout(() => yEl.classList.remove('dropping'), end+60);
}
$('yReplay').onclick = () => { yearsSeen = true; dropNext = true; renderYears(rows()); };
if(REDUCED) $('yReplay').hidden = true;
yEl.addEventListener('mousemove', e => { const d = e.target.closest('.yd'); if(d) APP.tip.show(tipHTML(BY_I.get(+d.dataset.i)), e.clientX, e.clientY); else APP.tip.hide(); });
yEl.addEventListener('mouseleave', () => APP.tip.hide());
yEl.addEventListener('click', e => { const d = e.target.closest('.yd'); if(d) openFicha(+d.dataset.i); });
new IntersectionObserver((en, ob) => { if(en[0].intersectionRatio > .3){ ob.disconnect(); yearsSeen = true; dropNext = true; renderYears(rows()); } }, {threshold:[0,.3,.6]}).observe(yEl);

/* ── ranking de organismos ── */
$('rkPct').onclick = () => { st.rankBy='pct'; $('rkPct').setAttribute('aria-pressed',true); $('rkMi').setAttribute('aria-pressed',false); renderRank(rows()); };
$('rkMi').onclick  = () => { st.rankBy='mi';  $('rkMi').setAttribute('aria-pressed',true); $('rkPct').setAttribute('aria-pressed',false); renderRank(rows()); };
function renderRank(arr){
  const g = new Map();
  arr.forEach(r => { if(!g.has(r.p)) g.set(r.p,{n:0,mi:0,c:{}}); const o=g.get(r.p); o.n++; o.mi+=r.mi; o.c[r.st]=(o.c[r.st]||0)+1; });
  let list = [...g].filter(([k,o])=>o.n>=5).map(([k,o])=>({k,o,bad:((o.c['Atrasado']||0)+(o.c['Abandonado']||0))/o.n}));
  list.sort(st.rankBy==='pct' ? (a,b)=>b.bad-a.bad||b.o.n-a.o.n : (a,b)=>b.o.mi-a.o.mi||b.bad-a.bad);
  list = list.slice(0,15);
  const maxMi = Math.max(1, ...list.map(x=>x.o.mi));
  $('rank').innerHTML = list.map(x =>
    '<button type="button" class="rk'+(st.org===x.k?' sel':'')+'" data-org="'+esc(x.k)+'"><span class="rn">'+esc(x.k)+' <small>('+x.o.n+')</small></span>'+
    '<span class="rb">'+ESTADOS.filter(e=>x.o.c[e]).map(e=>'<span data-e="'+e+'" data-v="'+x.o.c[e]+'" data-n="'+x.o.n+'" style="flex:'+x.o.c[e]+' 1 0;background:'+COLOR[e]+'"></span>').join('')+'</span>'+
    '<span class="rm"><i style="width:'+Math.max(2,70*x.o.mi/maxMi)+'px"></i>'+fmt(x.o.mi)+'</span></button>').join('') ||
    '<p class="small">Ningún organismo con 5 o más conjuntos coincide con los filtros.</p>';
}
$('rank').addEventListener('click', e => { const b = e.target.closest('.rk'); if(!b) return; st.org = st.org===b.dataset.org ? '' : b.dataset.org; syncControls(); changed(); });
$('rank').addEventListener('mousemove', e => {
  const s = e.target.closest('.rb span');
  if(s) APP.tip.show('<div class="t">'+s.dataset.e+'</div><div>'+s.dataset.v+' de '+s.dataset.n+' conjuntos ('+pct(+s.dataset.v,+s.dataset.n)+' %)</div>', e.clientX, e.clientY); else APP.tip.hide();
});
$('rank').addEventListener('mouseleave', () => APP.tip.hide());

/* ── tabla ordenable ── */
const COLS = [['t','Conjunto de datos'],['p','Organismo'],['st','Estado'],['age','Última act.'],['f','Frecuencia'],['mi','Faltan'],['c','Metadatos']];
function renderHead(){
  $('thead').innerHTML = COLS.map(([k,l]) => '<th><button type="button" data-k="'+k+'"'+(st.sort.k===k?' data-dir="'+(st.sort.dir>0?'↑':'↓')+'"':'')+'>'+l+'</button></th>').join('');
  $('thead').querySelectorAll('button').forEach(b => b.onclick = () => {
    const k = b.dataset.k; st.sort = {k, dir: st.sort.k===k ? -st.sort.dir : (['t','p','f'].includes(k)?1:-1)};
    renderHead(); renderTable(rows());
  });
}
function sortVal(r,k){ return k==='st' ? ORD[r.st] : (k==='age' ? (r.age==null?-1:r.age) : r[k]); }
/* resalta la búsqueda sin importar tildes ni mayúsculas (la normalización es letra por letra) */
function hl(text, q){
  text = String(text==null?'':text);
  if(!q) return esc(text);
  const n = [...text].map(c => norm(c) || c).join(''), at = n.indexOf(q);
  if(at < 0 || n.length !== text.length) return esc(text);
  return esc(text.slice(0,at))+'<mark>'+esc(text.slice(at,at+q.length))+'</mark>'+esc(text.slice(at+q.length));
}
$('tq').oninput = e => { st.tq = e.target.value; st.shown = 60; renderTable(rows()); };
function renderTable(arr){
  const {k,dir} = st.sort, tq = norm(st.tq.trim());
  if(tq) arr = arr.filter(r => (r._t || (r._t = norm(r.t+' · '+r.p+' · '+r.th+' · '+r.f))).includes(tq));
  const sorted = arr.slice().sort((a,b) => { const x=sortVal(a,k), y=sortVal(b,k); return (typeof x==='string' ? x.localeCompare(y,'es') : x-y)*dir; });
  $('tsum').textContent = '· '+fmt(arr.length)+' conjuntos'+(tq ? ' coinciden con «'+st.tq.trim()+'»' : '');
  $('tcount').textContent = (tq ? fmt(sorted.length)+' coinciden con «'+st.tq.trim()+'» · ' : '')+'mostrando '+fmt(Math.min(st.shown, sorted.length))+' de '+fmt(sorted.length);
  $('rows').innerHTML = sorted.slice(0, st.shown).map(r =>
    '<tr data-i="'+r.i+'"><td>'+hl(r.t,tq)+'</td><td>'+hl(r.p,tq)+'</td>'+
    '<td><span class="tag" style="background:'+COLOR[r.st]+'">'+r.st+'</span></td>'+
    '<td>'+fdate(r.d)+'<div class="small">'+fmtAge(r.age)+'</div></td><td>'+hl(r.f,tq)+'</td>'+
    '<td class="num">'+(r.pr ? fmt(r.mi) : '—')+'</td><td class="num">'+r.c+'/100</td></tr>').join('') ||
    '<tr><td colspan="7">'+(tq ? 'Ningún conjunto de la tabla coincide con «'+esc(st.tq.trim())+'».' : 'Ningún conjunto coincide. Probá quitar algún filtro.')+'</td></tr>';
  $('more').style.display = sorted.length>st.shown ? '' : 'none';
}
$('rows').addEventListener('click', e => { const tr = e.target.closest('tr[data-i]'); if(tr) openFicha(+tr.dataset.i); });
$('more').onclick = () => { st.shown += 60; renderTable(rows()); };
$('csv').onclick = () => {
  const cols = [['Título','t'],['Organismo','p'],['Tema','th'],['Estado','st'],['Última actualización','d'],['Años sin actualizar','age'],['Frecuencia','f'],['Actualizaciones que faltan','mi'],['Metadatos (0-100)','c'],['Provincias','pv'],['Enlace','u']];
  const q = v => '"'+String(Array.isArray(v)?v.join('; '):(v==null?'':v)).replace(/"/g,'""')+'"';
  const lines = [cols.map(c=>q(c[0])).join(',')].concat(rows().map(r=>cols.map(c=>q(r[c[1]])).join(',')));
  const a = document.createElement('a');
  a.href = URL.createObjectURL(new Blob(['﻿'+lines.join('\n')],{type:'text/csv;charset=utf-8'}));
  a.download = 'conjuntos_filtrados.csv'; document.body.appendChild(a); a.click(); a.remove(); setTimeout(()=>URL.revokeObjectURL(a.href), 1000);
};

/* ── ficha de cada conjunto ── */
const ficha = $('ficha'), scrim = $('scrim'); let lastFocus = null;
function promiseStrip(r){
  if(!r.pr) return '<p class="small">No declara cada cuánto se actualiza: no hay una promesa contra la cual medirlo.</p>';
  if(!r.d || !r.fd) return '';
  const W = 388, H = 64, x0 = 8, x1 = W-8, yL = 30;
  const last = new Date(r.d+'T00:00:00'), days = Math.max(1, (genDate-last)/864e5), fd = Math.max(r.fd,1);
  const X = t => x0 + (x1-x0)*Math.min(1, t/days);
  let s = '<svg class="strip" viewBox="0 0 '+W+' '+H+'" role="img" aria-label="Línea de tiempo desde la última actualización">';
  s += '<line x1="'+x0+'" x2="'+x1+'" y1="'+yL+'" y2="'+yL+'" stroke="#2C3D5A" stroke-width="2"/>';
  if(r.st==='Al día'){
    s += '<circle cx="'+x0+'" cy="'+yL+'" r="6" fill="#35B8A8"/>';
  } else if(r.mi <= 150){
    for(let k=1; k<=r.mi; k++) s += '<circle cx="'+X(k*fd).toFixed(1)+'" cy="'+yL+'" r="'+(r.mi>60?2.2:3.4)+'" fill="#111A2A" stroke="#E0507A" stroke-width="1.4"/>';
    s += '<circle cx="'+x0+'" cy="'+yL+'" r="6" fill="#35B8A8"/>';
  } else {
    s += '<defs><pattern id="hz" width="3" height="10" patternUnits="userSpaceOnUse"><rect width="1.4" height="10" fill="#E0507A"/></pattern></defs>';
    s += '<rect x="'+X(fd).toFixed(1)+'" y="'+(yL-7)+'" width="'+(x1-X(fd)).toFixed(1)+'" height="14" fill="url(#hz)" opacity=".85"/>';
    s += '<circle cx="'+x0+'" cy="'+yL+'" r="6" fill="#35B8A8"/>';
  }
  s += '<line x1="'+x1+'" x2="'+x1+'" y1="'+(yL-12)+'" y2="'+(yL+12)+'" stroke="#E7EDF4" stroke-width="2"/>';
  s += '<text x="'+x0+'" y="'+(H-6)+'" fill="#94A3B6" font-size="11">'+fdate(r.d)+'</text><text x="'+x1+'" y="'+(H-6)+'" fill="#94A3B6" font-size="11" text-anchor="end">hoy ('+META.generado+')</text>';
  s += '<text x="'+x0+'" y="12" fill="#94A3B6" font-size="11">última actualización</text>';
  if(r.st!=='Al día') s += '<text x="'+x1+'" y="12" fill="#E0507A" font-size="11" text-anchor="end">'+(r.mi<=150?'cada círculo, una que no llegó':'una marca por cada actualización que no llegó')+'</text>';
  s += '</svg>';
  if(r.st==='Al día'){ const nx = new Date(last.getTime()+fd*864e5); s += '<p class="small">Próxima actualización esperada: alrededor del '+nx.toLocaleDateString('es-AR')+'.</p>'; }
  return s;
}
function openFicha(i){
  const r = BY_I.get(i); if(!r) return;
  lastFocus = document.activeElement;
  let big;
  if(!r.pr) big = '<b class="k-nop">—</b><span>No promete ninguna frecuencia de actualización.'+(r.age>3?' Lleva '+fmtAge(r.age)+' sin cambios.':'')+'</span>';
  else if(r.st==='Al día') big = '<b class="k-ok">0</b><span>actualizaciones pendientes: cumple lo que promete.</span>';
  else big = '<b class="k-dead">'+fmt(r.mi)+'</b><span>actualizaciones prometidas que nunca llegaron. Prometía: '+esc(r.f).toLowerCase()+'.</span>';
  ficha.innerHTML =
    '<button class="ib x" type="button" id="fX" aria-label="Cerrar ficha">×</button>'+
    '<span class="tag" style="background:'+COLOR[r.st]+'">'+r.st+'</span>'+
    '<h3>'+esc(r.t)+'</h3><p class="org">'+esc(r.p)+' · '+esc(r.th)+'</p>'+
    '<div class="big">'+big+'</div>'+ promiseStrip(r)+
    '<dl class="facts"><dt>Última actualización</dt><dd>'+fdate(r.d)+' · hace '+fmtAge(r.age)+'</dd>'+
    '<dt>Frecuencia prometida</dt><dd>'+esc(r.f)+'</dd>'+
    '<dt>Recursos</dt><dd>'+r.n+(r.n===1?' archivo':' archivos')+'</dd>'+
    '<dt>Provincias</dt><dd>'+(r.pv.length ? esc(r.pv.join(', ')) : '<span class="small">No nombra ninguna</span>')+'</dd>'+
    '<dt>Metadatos</dt><dd><span class="meter"><i style="width:'+r.c+'%"></i></span>'+r.c+'/100</dd></dl>'+
    '<div class="acts">'+(r.u ? '<a class="btn primary" href="'+esc(r.u)+'" target="_blank" rel="noopener">Ver la ficha original en datos.gob.ar ↗</a>' : '')+
    '<button class="btn" type="button" id="fLoc">Ubicar en la nube 3D</button>'+
    '<button class="btn" type="button" id="fCity">Ver su edificio en la ciudad 3D</button>'+
    '<button class="btn" type="button" id="fOrg">'+(st.org===r.p?'Quitar el filtro de organismo':'Filtrar por este organismo')+'</button></div>';
  ficha.classList.add('open'); scrim.classList.add('open'); ficha.setAttribute('aria-hidden','false');
  $('fX').onclick = closeFicha;
  $('fLoc').onclick = () => { closeFicha(); locate(i); };
  $('fCity').onclick = () => { closeFicha(); if(st.mv!=='3d') setMosView('3d'); $('mosaico').scrollIntoView({behavior: REDUCED?'auto':'smooth', block:'start'}); setTimeout(()=>{ if(APP.city) APP.city.focus(i); }, 350); };
  $('fOrg').onclick = () => { st.org = st.org===r.p ? '' : r.p; syncControls(); changed(); closeFicha(); };
  if(APP.cloud) APP.cloud.select(i);
  if(APP.city) APP.city.select(i);
  setTimeout(()=>$('fX').focus(), 50);
}
function closeFicha(){
  ficha.classList.remove('open'); scrim.classList.remove('open'); ficha.setAttribute('aria-hidden','true');
  if(APP.city) APP.city.select(null);
  if(lastFocus && lastFocus.focus) lastFocus.focus();
}
APP.openFicha = openFicha;
scrim.onclick = closeFicha;
addEventListener('keydown', e => { if(e.key==='Escape' && ficha.classList.contains('open')) closeFicha(); });

/* ── nube 3D: el módulo se engancha acá ── */
function cloudLegend(base){
  const c = counts(base);
  $('cloudLegend').innerHTML = ESTADOS.filter(e=>c[e]>0).map(e=>'<span><span class="dot" style="background:'+COLOR[e]+'"></span>'+e+'</span>').join('');
}
APP.cloudReady = () => { $('cloudMsg').remove(); if(APP.cloud) APP.cloud.update(rows(), st.group); };
APP.cloudFail = () => { const m=$('cloudMsg'); if(m) m.textContent = 'La vista 3D necesita WebGL y conexión a internet para cargar la librería Three.js. El resto del tablero funciona igual.'; };
setTimeout(() => { if(!APP.cloud) APP.cloudFail(); }, 12000);

/* ── intro guiada ── */
const prom = DATA.filter(r=>r.pr===1), inc = prom.filter(isBad).length, totMi = DATA.reduce((s,r)=>s+r.mi,0);
const FEAT = (() => {
  const cand = DATA.filter(r=>r.pr===1 && isBad(r));
  return cand.filter(r=>/\bal dia\b/.test(norm(r.t))).sort((a,b)=>b.mi-a.mi)[0] || cand.sort((a,b)=>b.mi-a.mi)[0];
})();
APP.FEAT = FEAT ? FEAT.i : null;
const ADV = {'Diaria':'todos los días','Semanal':'todas las semanas','Quincenal':'cada quince días','Mensual':'todos los meses','Bimestral':'cada dos meses','Trimestral':'cada tres meses','Cuatrimestral':'cada cuatro meses','Semestral':'cada seis meses','Anual':'todos los años'};
const nOrg = new Set(DATA.map(r=>r.p)).size, nTema = new Set(DATA.map(r=>r.th)).size;
const maxAge = Math.round(Math.max(...DATA.map(r=>r.age||0)));
const medAge = median(DATA.filter(r=>r.age!=null).map(r=>r.age));
const C = e => '<span style="color:'+COLOR[e]+';font-weight:600">';
const STEPS = [
  {title:true, k:'Datos Argentina · metadatos al '+META.generado, h:'El mapa de lo que no se mide', p:'Qué pasó con la promesa de actualizar los '+fmt(N)+' conjuntos de datos abiertos del Estado nacional.'},
  {h:fmt(N)+' conjuntos de datos.', p:'Cada punto de luz es uno. Los publican <b>'+nOrg+' organismos</b> sobre <b>'+nTema+' temas</b>: estadísticas, precios, salud, energía, transporte.'},
  {h:'Cada organismo prometió cada cuánto los iba a actualizar.', p:'Acá están agrupados por organismo y pintados según si cumplen: '+C('Al día')+'al día</span>, '+C('Atrasado')+'atrasado</span>, '+C('Abandonado')+'abandonado</span> o '+C('Sin periodicidad')+'sin promesa</span> (no declara frecuencia).'},
  {h:'Cuanto más lejos del centro, más tiempo sin cambios.', p:'El anillo interior es hoy y el borde, '+maxAge+' años atrás. La mitad de los conjuntos lleva más de <b>'+fmt1(medAge)+' años</b> sin actualizarse.'},
  {h:'Cuanto más alto, más actualizaciones prometidas nunca llegaron.', p:'En total faltan <b>'+fmt(totMi)+'</b>. Cada una es un dato que el Estado se comprometió a publicar y no publicó.'},
  {h:pct(inc,prom.length)+' de cada 100 conjuntos que prometen actualizarse ya no lo hacen.', p:'Son <b>'+fmt(inc)+'</b> de '+fmt(prom.length)+'. Y otros <b>'+fmt(N-prom.length)+'</b> ni siquiera dicen cada cuánto se actualizan.'},
  FEAT ? {h:(/\bal dia\b/.test(norm(FEAT.t)) ? 'Uno de ellos se llama «'+esc(FEAT.t)+'».' : 'El caso extremo: «'+esc(FEAT.t)+'».'),
          p:'Lo publica '+esc(FEAT.p)+'. Prometía actualizarse <b>'+(ADV[FEAT.f]||esc(FEAT.f).toLowerCase())+'</b>. La última vez fue el '+fdate(FEAT.d)+'. Desde entonces faltan <b>'+fmt(FEAT.mi)+'</b> actualizaciones.'} : null,
  {h:'Ahora explorá vos.', p:'Filtrá por tema u organismo, buscá un conjunto o tocá cualquier punto para ver su ficha.', cta:true}
].filter(Boolean);
APP.STEP_KEYS = STEPS.map((s,k)=> s.title ? 'title' : (k===STEPS.length-1 ? 'end' : ['','count','groups','rings','height','bad','feat'][k]));
$('steps').innerHTML = STEPS.map((s,k) => '<div class="step'+(s.title?' title':'')+'" data-k="'+k+'"><div class="card">'+
  (s.title ? '<p class="kicker">'+s.k+'</p><h1>'+s.h+'</h1><p>'+s.p+'</p><div class="cue">Deslizá para empezar ↓</div>'
           : '<h2>'+s.h+'</h2><p>'+s.p+'</p>'+(s.cta?'<a class="cta" href="#tablero">Ir al tablero ↓</a>':''))+'</div></div>').join('');
$('dots').innerHTML = STEPS.map(()=>'<i></i>').join('');
APP.storyStep = 0;
const stepEls = [...document.querySelectorAll('.step')];
function goStep(k){
  APP.storyStep = k;
  stepEls.forEach((el,j)=>el.classList.toggle('on', j===k));
  [...$('dots').children].forEach((d,j)=>d.classList.toggle('on', j===k));
  if(APP.story) APP.story.go(k);
}
const io = new IntersectionObserver(en => { en.forEach(e => { if(e.isIntersecting) goStep(+e.target.dataset.k); }); }, {rootMargin:'-45% 0px -45% 0px'});
stepEls.forEach(el => io.observe(el));
goStep(0);

/* ── render general ── */
function render(){
  const arr = rows(), base = rows(true);
  renderHero(base); renderBar(base, arr); cloudLegend(base); cityLegend(base);
  renderMos(arr); renderYears(arr); renderRank(arr); renderTable(arr);
  const np = arr.filter(r=>r.pv.length).length;
  $('cloudNote').textContent = 'Se destacan '+fmt(arr.length)+' de '+fmt(N)+' conjuntos. Las actualizaciones que faltan van en escala logarítmica: cada nivel de altura multiplica por diez.';
  if(APP.cloud) APP.cloud.update(arr, st.group);
  writeHash();
}
APP.render = render;
let rz; addEventListener('resize', () => { clearTimeout(rz); rz = setTimeout(() => { const a = rows(); renderMos(a); renderYears(a); }, 150); });
const mqMobile = matchMedia('(max-width: 900px)');
if(mqMobile.matches) $('filtros').open = false;
mqMobile.addEventListener && mqMobile.addEventListener('change', e => { if(!e.matches) $('filtros').open = true; });
readHash(); syncControls(); renderHead();
$('mosLede').innerHTML = LEDE[st.mv]; $('mosTitle').textContent = st.mv==='3d' ? 'La ciudad del olvido' : 'Dónde se concentra el olvido';
$('city').hidden = st.mv!=='3d'; treeEl.hidden = st.mv!=='2d';
render();
setTimeout(() => { if(!APP.city) APP.cityFail(); }, 12000);
})();
</script>

<script type="module">
import * as THREE from 'three';
import { OrbitControls } from 'three/addons/controls/OrbitControls.js';

const APP = window.APP;
const { DATA, COLOR, REDUCED, N } = APP;
const J = new Map(DATA.map((r,j)=>[r.i,j]));
const rnd = (i,k) => { const x = Math.sin(i*12.9898 + k*78.233)*43758.5453; return x - Math.floor(x); };
const AGEMAX = Math.max(9, ...DATA.map(r=>r.age||0));
const R0 = 1.5, KR = 1.05, RMAX = R0 + KR*AGEMAX;
const HK = 1.75, hY = mi => Math.log10(1+mi)*HK;
const SIZE = {'Al día':1.35,'Atrasado':1.1,'Abandonado':.95,'Sin periodicidad':.85,'Sin fecha':.8};
const ALPHA = {'Al día':1,'Atrasado':1,'Abandonado':.9,'Sin periodicidad':.6,'Sin fecha':.5};
const NEUTRAL = new THREE.Color('#C8D3E0');
const STCOL = {}; Object.keys(COLOR).forEach(k => STCOL[k] = new THREE.Color(COLOR[k]));
const shortName = g => {
  const m = g.match(/\(([^)]{2,12})\)\s*$/); if(m) return m[1];
  const s = g.replace(/^Subsecretaría de /,'Subs. ').replace(/^Secretaría de /,'Sec. ').replace(/^Ministerio de /,'Min. ')
             .replace(/^Dirección Nacional de /,'DN ').replace(/^Instituto Nacional de /,'Inst. Nac. ').replace(/^Servicio Nacional de /,'Serv. Nac. ');
  return s.length > 30 ? s.slice(0,29)+'…' : s;
};

function layoutDisc(group, heights){
  const counts = new Map(); DATA.forEach(r => counts.set(r[group], (counts.get(r[group])||0)+1));
  const groups = [...counts].sort((a,b)=>b[1]-a[1]);
  const gap = 0.04, avail = Math.PI*2 - gap*groups.length;
  let a0 = -Math.PI/2 + gap/2; const wedges = new Map();
  groups.forEach(([g,n]) => { const w = avail*n/N; wedges.set(g,{a0, a1:a0+w, n}); a0 += w+gap; });
  const pos = new Float32Array(N*3);
  DATA.forEach((r,j) => {
    const w = wedges.get(r[group]), span = w.a1-w.a0, pad = Math.min(0.012, span*0.08);
    const a = w.a0 + pad + rnd(r.i,1)*(span-2*pad);
    const rad = R0 + KR*(r.age==null ? AGEMAX+0.5 : r.age) + (rnd(r.i,2)-.5)*0.16;
    pos[j*3] = Math.cos(a)*rad; pos[j*3+2] = Math.sin(a)*rad;
    pos[j*3+1] = (heights ? hY(r.mi) : 0) + (rnd(r.i,3)-.5)*0.08;
  });
  return {pos, wedges, groups};
}
function layoutSphere(){
  const pos = new Float32Array(N*3), inc = Math.PI*(3-Math.sqrt(5));
  DATA.forEach((r,j) => {
    const y = 1 - (j+.5)*2/N, rr = Math.sqrt(1-y*y), phi = j*inc, R = 6.2*(0.9+0.2*rnd(r.i,5));
    pos[j*3] = Math.cos(phi)*rr*R; pos[j*3+1] = y*R; pos[j*3+2] = Math.sin(phi)*rr*R;
  });
  return pos;
}

const VERT = `
attribute vec3 aColor; attribute float aAlpha; attribute float aSize; attribute float aPulse;
uniform float uScale; uniform float uPR; uniform float uTime;
varying vec3 vColor; varying float vAlpha;
void main(){
  vColor = aColor; vAlpha = aAlpha;
  vec4 mv = modelViewMatrix * vec4(position,1.0);
  float pulse = 1.0 + aPulse*0.22*sin(uTime*2.4 + position.x*1.7 + position.z*1.3);
  gl_PointSize = min(aSize * pulse * uScale * uPR / -mv.z, 72.0*uPR);
  gl_Position = projectionMatrix * mv;
}`;
const FRAG = `
varying vec3 vColor; varying float vAlpha;
void main(){
  float d = length(gl_PointCoord - 0.5);
  if(d > 0.5) discard;
  float core = smoothstep(0.2, 0.1, d);
  float glow = pow(1.0 - d*2.0, 2.2);
  vec3 col = vColor*(0.55*glow + 0.95*core) + vec3(1.0)*core*0.18;
  gl_FragColor = vec4(col, (core + glow*0.6)*vAlpha);
}`;

class Cloud {
  constructor(el, interactive){
    this.el = el; this.interactive = interactive;
    const R = this.renderer = new THREE.WebGLRenderer({antialias:true, alpha:true});
    R.setPixelRatio(Math.min(devicePixelRatio, 2)); R.setClearColor(0x000000, 0);
    el.appendChild(R.domElement);
    this.scene = new THREE.Scene();
    this.camera = new THREE.PerspectiveCamera(42, 1, 0.1, 300);
    this.camera.position.set(interactive ? 19 : 15, interactive ? 14 : 11, interactive ? 19 : 15);
    this.target = new THREE.Vector3(0, 1, 0);
    this.world = new THREE.Group(); this.scene.add(this.world);

    const floor = new THREE.Mesh(new THREE.CircleGeometry(RMAX+1.4, 96), new THREE.MeshBasicMaterial({color:0x101C30, transparent:true, opacity:.6, depthWrite:false}));
    floor.rotation.x = -Math.PI/2; floor.position.y = -0.06; this.world.add(floor);
    this.floorMat = floor.material;
    this.guides = new THREE.Group(); this.world.add(this.guides);

    this.cur = new Float32Array(N*3); this.tgt = new Float32Array(N*3);
    this.col = new Float32Array(N*3); this.colT = new Float32Array(N*3);
    this.alpha = new Float32Array(N); this.alphaT = new Float32Array(N);
    this.size = new Float32Array(N); this.sizeT = new Float32Array(N);
    this.pulse = new Float32Array(N);
    const sph = layoutSphere(); this.cur.set(sph); this.tgt.set(sph);
    DATA.forEach((r,j) => { this.size[j] = SIZE[r.st]*.2; this.pulse[j] = r.st==='Al día' ? 1 : 0; NEUTRAL.toArray(this.col, j*3); });
    const geo = this.geo = new THREE.BufferGeometry();
    geo.setAttribute('position', new THREE.BufferAttribute(this.cur, 3));
    geo.setAttribute('aColor', new THREE.BufferAttribute(this.col, 3));
    geo.setAttribute('aAlpha', new THREE.BufferAttribute(this.alpha, 1));
    geo.setAttribute('aSize', new THREE.BufferAttribute(this.size, 1));
    geo.setAttribute('aPulse', new THREE.BufferAttribute(this.pulse, 1));
    this.mat = new THREE.ShaderMaterial({vertexShader:VERT, fragmentShader:FRAG, transparent:true, depthWrite:false, blending:THREE.AdditiveBlending,
      uniforms:{uScale:{value:800}, uPR:{value:R.getPixelRatio()}, uTime:{value:0}}});
    this.points = new THREE.Points(geo, this.mat); this.points.frustumCulled = false; this.world.add(this.points);

    this.lbl = document.createElement('div'); this.lbl.className = 'lbls'; el.appendChild(this.lbl);
    this.labels = [];
    this.markEl = document.createElement('div'); this.markEl.className = 'mark'; el.appendChild(this.markEl);
    this.sel = null; this.hover = null; this.group = null; this.heights = false;
    this.show = {rings:false, axis:false, groups:false, guides:false};
    this.guideAlpha = 0; this.colorMode = 'neutral'; this.vis = null; this.dimFn = null;
    this.rotSpeed = 0; this.tw = null; this.visible = true;
    this.v = new THREE.Vector3();

    if(interactive){
      const c = this.controls = new OrbitControls(this.camera, R.domElement);
      c.enableDamping = true; c.dampingFactor = .08; c.enablePan = false; c.enableZoom = false;
      c.minDistance = 3; c.maxDistance = 48; c.maxPolarAngle = Math.PI*0.495; c.rotateSpeed = .6;
      c.autoRotate = !REDUCED; c.autoRotateSpeed = .45; c.target.copy(this.target);
      R.domElement.addEventListener('pointerdown', e => { c.enableZoom = true; this.down = [e.clientX, e.clientY]; this.stopSpin(); });
      R.domElement.addEventListener('pointerleave', () => { c.enableZoom = false; this.setHover(null); APP.tip.hide(); });
      R.domElement.addEventListener('pointermove', e => this.pick(e));
      R.domElement.addEventListener('pointerup', e => {
        if(!this.down || Math.hypot(e.clientX-this.down[0], e.clientY-this.down[1]) > 6) return;
        this.pick(e);
        if(this.hover!=null){ const i = DATA[this.hover].i; this.focus(i); APP.openFicha(i); }
      });
      R.domElement.addEventListener('wheel', () => this.stopSpin(), {passive:true});
    }

    new ResizeObserver(() => this.resize()).observe(el); this.resize();
    new IntersectionObserver(en => { this.visible = en[0].isIntersecting; }).observe(el);
    this.clock = new THREE.Clock();
    const loop = () => { requestAnimationFrame(loop); if(this.visible) this.frame(); };
    loop();
  }
  resize(){
    const w = this.el.clientWidth, h = this.el.clientHeight; if(!w || !h) return;
    this.renderer.setSize(w, h, false); this.camera.aspect = w/h;
    if(!this.interactive && w > 900) this.camera.setViewOffset(w, h, -w*0.17, 0, w, h); else this.camera.clearViewOffset();
    this.camera.updateProjectionMatrix();
    this.mat.uniforms.uScale.value = h / (2*Math.tan(THREE.MathUtils.degToRad(this.camera.fov/2))) * 0.36;
    this.w = w; this.h = h;
  }
  stopSpin(){ if(this.controls && this.controls.autoRotate){ this.controls.autoRotate = false; const b = document.getElementById('spin'); if(b) b.setAttribute('aria-pressed','false'); } }

  /* disposición */
  setLayout(kind, group, heights){
    if(kind==='sphere'){ this.tgt.set(layoutSphere()); this.layoutKind='sphere'; return; }
    if(this.layoutKind==='disc' && this.group===group && this.heights===heights) return;
    const L = layoutDisc(group, heights); this.tgt.set(L.pos);
    const regroup = this.group !== group || this.layoutKind!=='disc';
    this.layoutKind = 'disc'; this.group = group; this.heights = heights;
    if(regroup) this.buildGuides(L);
  }
  buildGuides(L){
    this.guides.clear(); this.labels.forEach(l=>l.el.remove()); this.labels = [];
    const mat = this.lineMat = new THREE.LineBasicMaterial({color:0x51699A, transparent:true, opacity:0});
    const circle = r => { const p=[]; for(let k=0;k<=128;k++){ const a=k/128*Math.PI*2; p.push(new THREE.Vector3(Math.cos(a)*r,0,Math.sin(a)*r)); } return new THREE.Line(new THREE.BufferGeometry().setFromPoints(p), mat); };
    const firstA = L.wedges.get(L.groups[0][0]).a0 - 0.02;
    const ages = [0,1,3,5,7,9].filter(a=>a<=AGEMAX);
    ages.forEach(a => {
      this.guides.add(circle(R0+KR*a));
      const ang = firstA;
      this.addLabel(a===0 ? 'hoy' : (a===1 ? '1 año' : a+' años'), new THREE.Vector3(Math.cos(ang)*(R0+KR*a), 0.05, Math.sin(ang)*(R0+KR*a)), 'r');
    });
    const seg = [];
    L.groups.forEach(([g]) => { const w = L.wedges.get(g), a = w.a0 - 0.02; seg.push(new THREE.Vector3(Math.cos(a)*R0,0,Math.sin(a)*R0), new THREE.Vector3(Math.cos(a)*(RMAX+.5),0,Math.sin(a)*(RMAX+.5))); });
    this.guides.add(new THREE.LineSegments(new THREE.BufferGeometry().setFromPoints(seg), mat));
    const axMat = this.axMat = new THREE.LineBasicMaterial({color:0x4A5F82, transparent:true, opacity:0});
    const top = hY(Math.max(...DATA.map(r=>r.mi))) + .3, ax = [new THREE.Vector3(0,0,0), new THREE.Vector3(0,top,0)];
    [10,100,1000].forEach(v => { const y = hY(v); ax.push(new THREE.Vector3(-.18,y,0), new THREE.Vector3(.18,y,0)); this.addLabel(v.toLocaleString('es-AR')+(v===1000?' actualizaciones que faltan':''), new THREE.Vector3(.3,y,0), 'ax', 'left'); });
    this.axis = new THREE.LineSegments(new THREE.BufferGeometry().setFromPoints([ax[0],ax[1]].concat(ax.slice(2))), axMat); this.guides.add(this.axis);
    const nShow = this.group==='p' ? 11 : 99;
    L.groups.forEach(([g,n],k) => {
      const w = L.wedges.get(g), mid = (w.a0+w.a1)/2, rr = RMAX + 1.1;
      const l = this.addLabel(shortName(g)+' · '+n, new THREE.Vector3(Math.cos(mid)*rr, 0.1, Math.sin(mid)*rr), 'g');
      l.minor = k >= nShow; l.g = g;
    });
  }
  addLabel(text, v, kind, align){
    const el = document.createElement('div'); el.className = 'lbl '+kind; el.textContent = text; this.lbl.appendChild(el);
    const l = {el, v, kind, align}; this.labels.push(l); return l;
  }
  /* color y visibilidad */
  applyStyle(){
    DATA.forEach((r,j) => {
      const c = this.colorMode==='neutral' ? NEUTRAL : STCOL[r.st];
      this.colT[j*3]=c.r; this.colT[j*3+1]=c.g; this.colT[j*3+2]=c.b;
      const on = (!this.vis || this.vis.has(r.i)) && (!this.dimFn || !this.dimFn(r));
      let s = SIZE[r.st]*(on?1:.55);
      if(j===this.hover) s *= 1.9; else if(r.i===this.sel) s *= 1.6;
      this.sizeT[j] = s;
      this.alphaT[j] = on ? (this.colorMode==='neutral' ? .6 : ALPHA[r.st]) : .05;
    });
  }
  setHover(j){ if(j===this.hover) return; this.hover = j; this.applyStyle(); this.renderer.domElement.style.cursor = j==null ? 'grab' : 'pointer'; }
  pick(e){
    const rect = this.renderer.domElement.getBoundingClientRect(), mx = e.clientX-rect.left, my = e.clientY-rect.top;
    this.world.updateMatrixWorld();
    let best = null, bd = 15*15;
    for(let j=0;j<N;j++){
      if(this.alphaT[j] < .3) continue;
      this.v.set(this.cur[j*3], this.cur[j*3+1], this.cur[j*3+2]).applyMatrix4(this.world.matrixWorld).project(this.camera);
      if(this.v.z > 1) continue;
      const dx = (this.v.x+1)/2*this.w - mx, dy = (1-this.v.y)/2*this.h - my, d = dx*dx+dy*dy;
      if(d < bd){ bd = d; best = j; }
    }
    this.setHover(best);
    if(best!=null) APP.tip.show(APP.tipHTML(DATA[best]), e.clientX, e.clientY); else APP.tip.hide();
  }
  /* cámara */
  camTo(pos, tgt, dur){
    const to = new THREE.Vector3(...pos), tt = new THREE.Vector3(...tgt);
    const ct = this.controls ? this.controls.target : this.target;
    if(REDUCED || !dur){ this.camera.position.copy(to); ct.copy(tt); this.tw = null; return; }
    this.tw = {p0:this.camera.position.clone(), p1:to, t0:ct.clone(), t1:tt, s:performance.now(), d:dur};
  }
  worldPos(j){ this.world.updateMatrixWorld(); return new THREE.Vector3(this.tgt[j*3], this.tgt[j*3+1], this.tgt[j*3+2]).applyMatrix4(this.world.matrixWorld); }
  focus(i){
    const j = J.get(i); if(j==null) return;
    this.stopSpin(); this.select(i);
    const p = this.worldPos(j), out = new THREE.Vector3(p.x, 0, p.z).normalize();
    if(!isFinite(out.x)) out.set(1,0,0);
    const cam = p.clone().add(out.multiplyScalar(5.5)).add(new THREE.Vector3(0, 3.2, 0));
    this.camTo([cam.x,cam.y,cam.z], [p.x,p.y,p.z], 1600);
  }
  select(i){ this.sel = i; this.applyStyle(); }
  view(kind){
    this.stopSpin();
    const V = {'3d':[[19,14,19],[0,1,0]], top:[[0,34,0.01],[0,0,0]], side:[[0,3.2,29],[0,2.2,0]]}[kind];
    this.camTo(V[0], V[1], 1400);
  }
  dolly(f){
    const c = this.controls; const off = this.camera.position.clone().sub(c.target).multiplyScalar(f);
    const d = Math.min(c.maxDistance, Math.max(c.minDistance, off.length())); off.setLength(d);
    const p = c.target.clone().add(off); this.camTo([p.x,p.y,p.z], [c.target.x,c.target.y,c.target.z], 500);
  }
  /* cuadro por cuadro */
  frame(){
    const dt = Math.min(this.clock.getDelta(), .05), t = this.clock.elapsedTime;
    this.mat.uniforms.uTime.value = REDUCED ? 0 : t;
    const kp = REDUCED ? 1 : 1 - Math.exp(-dt*3.2), kc = REDUCED ? 1 : 1 - Math.exp(-dt*4);
    for(let q=0;q<N*3;q++){ this.cur[q] += (this.tgt[q]-this.cur[q])*kp; this.col[q] += (this.colT[q]-this.col[q])*kc; }
    for(let j=0;j<N;j++){ this.alpha[j] += (this.alphaT[j]-this.alpha[j])*kc; this.size[j] += (this.sizeT[j]-this.size[j])*kc*1.5; }
    const A = this.geo.attributes; A.position.needsUpdate = A.aColor.needsUpdate = A.aAlpha.needsUpdate = A.aSize.needsUpdate = true;
    const ga = this.show.guides ? 1 : 0; this.guideAlpha += (ga-this.guideAlpha)*kc;
    if(this.lineMat) this.lineMat.opacity = .55*this.guideAlpha*(this.show.rings?1:.5);
    if(this.axMat) this.axMat.opacity = .8*this.guideAlpha*(this.show.axis?1:0);
    this.floorMat.opacity = .6*this.guideAlpha;
    if(this.rotSpeed) this.world.rotation.y += dt*this.rotSpeed;
    const ct = this.controls ? this.controls.target : this.target;
    if(this.tw){
      const k = Math.min(1, (performance.now()-this.tw.s)/this.tw.d), e = k<.5 ? 4*k*k*k : 1-Math.pow(-2*k+2,3)/2;
      this.camera.position.lerpVectors(this.tw.p0, this.tw.p1, e); ct.lerpVectors(this.tw.t0, this.tw.t1, e);
      if(k>=1) this.tw = null;
    }
    if(this.controls) this.controls.update(); else this.camera.lookAt(this.target);
    this.renderer.render(this.scene, this.camera);
    this.drawLabels();
  }
  drawLabels(){
    this.world.updateMatrixWorld();
    const cam = this.camera, w = this.w, h = this.h, M = this.world.matrixWorld;
    this.labels.forEach(l => {
      let on = this.guideAlpha > .5 && ((l.kind==='r' && this.show.rings) || (l.kind==='ax' && this.show.axis) || (l.kind==='g' && this.show.groups && (!l.minor || this.hoverGroup===l.g)));
      if(on){
        this.v.copy(l.v).applyMatrix4(M).project(cam);
        if(this.v.z > 1 || Math.abs(this.v.x) > 1.1 || Math.abs(this.v.y) > 1.1) on = false;
        else { const x = (this.v.x+1)/2*w, y = (1-this.v.y)/2*h; l.el.style.transform = 'translate('+x.toFixed(1)+'px,'+y.toFixed(1)+'px) translate('+(l.align==='left'?'0':'-50%')+',-50%)'; }
      }
      l.el.style.opacity = on ? 1 : 0;
    });
    const sj = this.sel==null ? null : J.get(this.sel);
    if(sj!=null && this.alphaT[sj] > .3){
      this.v.set(this.cur[sj*3], this.cur[sj*3+1], this.cur[sj*3+2]).applyMatrix4(M).project(cam);
      if(this.v.z < 1){ this.markEl.style.transform = 'translate('+((this.v.x+1)/2*w).toFixed(1)+'px,'+((1-this.v.y)/2*h).toFixed(1)+'px)'; this.markEl.classList.add('on'); }
      else this.markEl.classList.remove('on');
    } else this.markEl.classList.remove('on');
  }
  /* API del tablero */
  update(arr, group){
    this.vis = new Set(arr.map(r=>r.i)); this.colorMode = 'state';
    this.show = {rings:true, axis:true, groups:true, guides:true};
    this.setLayout('disc', group, true); this.applyStyle();
  }
  /* pasos de la intro */
  go(k){
    const key = APP.STEP_KEYS[k] || 'end';
    const S = {
      title:  {lay:'sphere', col:'neutral', cam:[0,1.5,22], tgt:[0,0,0], rot:.12},
      count:  {lay:'sphere', col:'neutral', cam:[0,0.5,16.5], tgt:[0,0,0], rot:.16},
      groups: {lay:'disc', h:false, col:'state', show:{guides:true, groups:true}, cam:[0,25,20], tgt:[0,0,0], rot:.05},
      rings:  {lay:'disc', h:false, col:'state', show:{guides:true, rings:true, groups:true}, cam:[0,37,0.01], tgt:[0,0,0], rot:.03},
      height: {lay:'disc', h:true,  col:'state', show:{guides:true, rings:true, axis:true}, cam:[0,3.4,27], tgt:[0,2.2,0], rot:.06},
      bad:    {lay:'disc', h:true,  col:'state', show:{guides:true, rings:true, axis:true}, dim:r=>!(r.pr===1 && (r.st==='Atrasado'||r.st==='Abandonado')), cam:[17,12,17], tgt:[0,1.2,0], rot:.05},
      feat:   {lay:'disc', h:true,  col:'state', show:{guides:true, rings:true, axis:true}, feat:true, rot:0},
      end:    {lay:'disc', h:true,  col:'state', show:{guides:true, rings:true, axis:true, groups:true}, cam:[16,12,16], tgt:[0,1.2,0], rot:.07}
    }[key];
    if(!S) return;
    this.setLayout(S.lay, 'p', !!S.h);
    this.colorMode = S.col; this.dimFn = S.dim || null; this.vis = null;
    this.show = Object.assign({rings:false, axis:false, groups:false, guides:false}, S.show||{});
    this.rotSpeed = REDUCED ? 0 : S.rot;
    this.sel = null;
    if(S.feat && APP.FEAT!=null){
      const j = J.get(APP.FEAT), p = this.worldPos(j), out = new THREE.Vector3(p.x,0,p.z).normalize();
      const cam = p.clone().add(out.multiplyScalar(6.5)).add(new THREE.Vector3(0,2.6,0));
      this.sel = APP.FEAT; this.dimFn = r => r.i!==APP.FEAT && r.p!==DATA[j].p;
      this.camTo([cam.x,cam.y,cam.z],[p.x,p.y,p.z], 1800);
    } else this.camTo(S.cam, S.tgt, 1800);
    this.applyStyle();
  }
}

/* ── la ciudad del olvido: una manzana por grupo, un edificio por conjunto ── */
const CELL = 1.25, FOOT = .86, GAPB = 1.9, HC = 1.15;
const LIT = {'Al día':.85,'Atrasado':.35,'Abandonado':.04,'Sin periodicidad':.14,'Sin fecha':.04};
const easeIO = k => k<.5 ? 4*k*k*k : 1-Math.pow(-2*k+2,3)/2;
class City {
  constructor(el){
    this.el = el;
    const R = this.renderer = new THREE.WebGLRenderer({antialias:true, alpha:true});
    R.setPixelRatio(Math.min(devicePixelRatio, 2)); R.setClearColor(0x000000, 0);
    el.appendChild(R.domElement);
    this.scene = new THREE.Scene(); this.scene.fog = new THREE.Fog(0x0A101C, 70, 190);
    this.camera = new THREE.PerspectiveCamera(38, 1, 0.1, 600);
    this.scene.add(new THREE.HemisphereLight(0xC9DAF5, 0x0B1322, 1.15));
    const sun = new THREE.DirectionalLight(0xFFFFFF, 1.7); sun.position.set(-30, 60, 25); this.scene.add(sun);
    const rim = new THREE.DirectionalLight(0x7FA2DA, .55); rim.position.set(40, 18, -35); this.scene.add(rim);
    const ground = new THREE.Mesh(new THREE.CircleGeometry(260, 64), new THREE.MeshBasicMaterial({color:0x0C1526}));
    ground.rotation.x = -Math.PI/2; ground.position.y = -0.03; this.scene.add(ground);

    const geo = new THREE.BoxGeometry(1,1,1); geo.translate(0,.5,0);
    this.lit = new Float32Array(N); DATA.forEach((r,j) => this.lit[j] = LIT[r.st]);
    geo.setAttribute('aLit', new THREE.InstancedBufferAttribute(this.lit, 1));
    const mat = new THREE.MeshStandardMaterial({roughness:.82, metalness:.04});
    mat.onBeforeCompile = sh => {
      sh.vertexShader = sh.vertexShader
        .replace('#include <common>', '#include <common>\nattribute float aLit; varying float vLit; varying vec3 vWP; varying vec3 vON;')
        .replace('#include <begin_vertex>', '#include <begin_vertex>\nvec4 wp_ = vec4(transformed, 1.0);\n#ifdef USE_INSTANCING\nwp_ = instanceMatrix * wp_;\n#endif\nvWP = (modelMatrix * wp_).xyz; vLit = aLit; vON = normal;');
      sh.fragmentShader = sh.fragmentShader
        .replace('#include <common>', '#include <common>\nvarying float vLit; varying vec3 vWP; varying vec3 vON;')
        .replace('#include <emissivemap_fragment>', `#include <emissivemap_fragment>
          if(abs(vON.y) < .5){
            float u_ = abs(vON.x) > .5 ? vWP.z : vWP.x;
            vec2 c_ = vec2(floor(u_*4.0), floor(vWP.y*3.2));
            float w_ = step(.3, fract(u_*4.0)) * step(fract(u_*4.0), .72) * step(.3, fract(vWP.y*3.2)) * step(fract(vWP.y*3.2), .74) * step(.25, vWP.y);
            float h_ = fract(sin(dot(c_ + vec2(floor(vWP.x*0.8)*7.0, floor(vWP.z*0.8)*13.0), vec2(12.9898, 78.233))) * 43758.5453);
            diffuseColor.rgb *= 1.0 - w_*.55;
            totalEmissiveRadiance += w_ * step(h_, vLit) * vec3(1.0, .9, .68) * .95;
          }`);
    };
    const mesh = this.mesh = new THREE.InstancedMesh(geo, mat, N);
    mesh.instanceMatrix.setUsage(THREE.DynamicDrawUsage);
    mesh.frustumCulled = false; mesh.boundingSphere = new THREE.Sphere(new THREE.Vector3(), 1e4);
    DATA.forEach((r,j) => mesh.setColorAt(j, STCOL[r.st]));
    mesh.instanceColor.needsUpdate = true;
    this.scene.add(mesh);
    this.plates = new THREE.Group(); this.scene.add(this.plates);
    this.plateMat = new THREE.MeshStandardMaterial({color:0x1A2944, roughness:1});
    this.plateHot = new THREE.MeshStandardMaterial({color:0x2C4268, roughness:1});
    this.selBox = new THREE.LineSegments(new THREE.EdgesGeometry(geo), new THREE.LineBasicMaterial({color:0xFFFFFF}));
    this.selBox.visible = false; this.scene.add(this.selBox);

    const A = () => new Float32Array(N);
    this.fx=A(); this.fz=A(); this.fh=A(); this.tx=A(); this.tz=A(); this.th=A(); this.dl=A();
    this.t0 = 0; this.revealed = false; this.anim = true; this.sig = ''; this.sel = null; this.hover = null; this.hoverBlock = null;
    this.m4 = new THREE.Matrix4(); this.q = new THREE.Quaternion(); this.vp = new THREE.Vector3(); this.vs = new THREE.Vector3(); this.v = new THREE.Vector3();
    this.blocks = []; this.extent = 20;

    this.lbl = document.createElement('div'); this.lbl.className = 'lbls'; el.appendChild(this.lbl);
    this.labels = [];
    this.ray = new THREE.Raycaster(); this.ndc = new THREE.Vector2();

    const c = this.controls = new OrbitControls(this.camera, R.domElement);
    c.enableDamping = true; c.dampingFactor = .08; c.enableZoom = false; c.screenSpacePanning = false;
    c.minDistance = 4; c.maxDistance = 240; c.maxPolarAngle = 1.38; c.rotateSpeed = .6;
    c.autoRotate = !REDUCED; c.autoRotateSpeed = .35;
    R.domElement.addEventListener('pointerdown', e => { c.enableZoom = true; this.down = [e.clientX, e.clientY]; this.stopSpin(); });
    R.domElement.addEventListener('pointerleave', () => { c.enableZoom = false; this.setHover(null, null); APP.tip.hide(); });
    R.domElement.addEventListener('pointermove', e => this.pick(e));
    R.domElement.addEventListener('wheel', () => this.stopSpin(), {passive:true});
    R.domElement.addEventListener('pointerup', e => {
      if(!this.down || Math.hypot(e.clientX-this.down[0], e.clientY-this.down[1]) > 6) return;
      this.pick(e);
      if(this.hover != null){ const i = DATA[this.hover].i; APP.openFicha(i); this.focus(i); }
      else if(this.hoverBlock){ APP.tip.hide(); APP.filterBlock(this.hoverBlock.name); }
    });

    new ResizeObserver(() => this.resize()).observe(el); this.resize();
    new IntersectionObserver(en => {
      this.visible = en[0].isIntersecting;
      if(this.visible && !this.revealed){ this.revealed = true; this.t0 = performance.now(); this.anim = true; }
    }, {threshold:.25}).observe(el);
    const loop = () => { requestAnimationFrame(loop); if(this.visible && this.w) this.frame(); };
    loop();
  }
  resize(){
    const w = this.el.clientWidth, h = this.el.clientHeight; if(!w || !h){ this.w = 0; return; }
    const first = !this.w;
    this.renderer.setSize(w, h, false); this.camera.aspect = w/h; this.camera.updateProjectionMatrix(); this.w = w; this.h = h;
    if(first && !this.homed){ this.homed = true; this.home(0); }
  }
  stopSpin(){ if(this.controls.autoRotate){ this.controls.autoRotate = false; const b = document.getElementById('cSpin'); if(b) b.setAttribute('aria-pressed','false'); } }
  homePose(){
    const asp = (this.w||16)/(this.h||9), d = this.extent*1.05*Math.max(1, 1.5/asp) + 6, yaw = .62, pitch = .6;
    return [[Math.sin(yaw)*Math.cos(pitch)*d, Math.sin(pitch)*d + 1.2, Math.cos(yaw)*Math.cos(pitch)*d], [0, 1.2, 0]];
  }
  home(dur){ const [p,t] = this.homePose(); this.camTo(p, t, dur==null ? 1100 : dur); }
  camTo(pos, tgt, dur){
    const to = new THREE.Vector3(...pos), tt = new THREE.Vector3(...tgt);
    if(REDUCED || !dur){ this.camera.position.copy(to); this.controls.target.copy(tt); this.tw = null; return; }
    this.tw = {p0:this.camera.position.clone(), p1:to, t0:this.controls.target.clone(), t1:tt, s:performance.now(), d:dur};
  }
  dolly(f){
    const c = this.controls, off = this.camera.position.clone().sub(c.target).multiplyScalar(f);
    off.setLength(Math.min(c.maxDistance, Math.max(c.minDistance, off.length())));
    const p = c.target.clone().add(off); this.camTo([p.x,p.y,p.z], [c.target.x,c.target.y,c.target.z], 450);
  }
  /* estado animado de cada edificio */
  cur(j, now){
    const k = !this.revealed ? 0 : Math.min(1, Math.max(0, (now - this.t0 - this.dl[j]) / 900)), e = easeIO(k);
    return [this.fx[j]+(this.tx[j]-this.fx[j])*e, this.fz[j]+(this.tz[j]-this.fz[j])*e, this.fh[j]+(this.th[j]-this.fh[j])*e];
  }
  layout(arr, key){
    const g = new Map(); arr.forEach(r => { if(!g.has(r[key])) g.set(r[key], []); g.get(r[key]).push(r); });
    const list = [...g].sort((a,b) => b[1].length - a[1].length);
    const area = list.reduce((s,[,a]) => s + Math.pow(Math.ceil(Math.sqrt(a.length))*CELL + GAPB, 2), 0);
    const maxW = Math.max(Math.sqrt(area)*1.35, list.length ? Math.ceil(Math.sqrt(list[0][1].length))*CELL : 1);
    const blocks = [], pos = new Map(); let x = 0, z = 0, rowD = 0, W = 0;
    list.forEach(([name, a]) => {
      const sd = Math.ceil(Math.sqrt(a.length)), size = sd*CELL;
      if(x > 0 && x + size > maxW){ x = 0; z += rowD + GAPB; rowD = 0; }
      const sorted = a.slice().sort((p,q) => (q.age||0) - (p.age||0));
      blocks.push({name, n:a.length, ab:a.filter(r=>r.st==='Abandonado').length, mi:a.reduce((s,r)=>s+r.mi,0), x0:x, z0:z, size});
      sorted.forEach((r,j) => pos.set(r.i, {x: x + (j%sd)*CELL + CELL/2, z: z + Math.floor(j/sd)*CELL + CELL/2, h: Math.max(.12, (r.age==null ? 0 : r.age)*HC)}));
      x += size + GAPB; rowD = Math.max(rowD, size); W = Math.max(W, x - GAPB);
    });
    const ox = W/2, oz = (z + rowD)/2;
    blocks.forEach(b => { b.x0 -= ox; b.z0 -= oz; }); pos.forEach(p => { p.x -= ox; p.z -= oz; });
    return {blocks, pos, extent: Math.max(W, z + rowD, 6)};
  }
  update(arr, group){
    const sig = group+'|'+arr.map(r=>r.i).join(',');
    if(sig === this.sig) return;
    const regroup = !this.sig || this.group !== group; this.sig = sig; this.group = group;
    const L = this.layout(arr, group), now = performance.now();
    DATA.forEach((r,j) => {
      const [cx, cz, ch] = this.cur(j, now), t = L.pos.get(r.i);
      this.fx[j] = cx; this.fz[j] = cz; this.fh[j] = ch;
      if(t){
        if(ch < .02){ this.fx[j] = t.x; this.fz[j] = t.z; }
        this.tx[j] = t.x; this.tz[j] = t.z; this.th[j] = t.h; this.dl[j] = ch < .02 ? rnd(r.i,7)*520 : 0;
      } else { this.tx[j] = cx; this.tz[j] = cz; this.th[j] = 0; this.dl[j] = 0; }
    });
    if(this.revealed) this.t0 = now;
    this.anim = true;
    this.blocks = L.blocks; this.buildPlates();
    const ext = L.extent; this.extent = ext;
    if(this.w && (regroup || Math.abs(ext - (this.lastExt||0)) > 2)) this.home(900);
    this.lastExt = ext;
  }
  buildPlates(){
    this.plates.children.forEach(m => m.geometry.dispose()); this.plates.clear();
    this.labels.forEach(l => l.el.remove()); this.labels = [];
    const m = .35;
    this.blocks.forEach(b => {
      const s = b.size + 2*m, plate = new THREE.Mesh(new THREE.BoxGeometry(s, .1, s), this.plateMat);
      plate.position.set(b.x0 + b.size/2, .05 - .1, b.z0 + b.size/2); plate.userData.block = b; this.plates.add(plate);
      const el = document.createElement('div'); el.className = 'lbl g';
      el.textContent = (this.group==='p' ? shortName(b.name) : b.name) + ' · ' + b.n; this.lbl.appendChild(el);
      this.labels.push({el, b, v: new THREE.Vector3(b.x0 - m, 0, b.z0 + b.size + m + .5)});
    });
  }
  setHover(j, b){
    if(j === this.hover && b === this.hoverBlock) return;
    this.hover = j; this.hoverBlock = b;
    this.plates.children.forEach(p => p.material = p.userData.block === b ? this.plateHot : this.plateMat);
    this.renderer.domElement.style.cursor = (j!=null || b) ? 'pointer' : 'grab';
  }
  pick(e){
    const rect = this.renderer.domElement.getBoundingClientRect();
    this.ndc.set((e.clientX-rect.left)/rect.width*2-1, -(e.clientY-rect.top)/rect.height*2+1);
    this.ray.setFromCamera(this.ndc, this.camera);
    const hit = this.ray.intersectObjects([this.mesh, ...this.plates.children], false)[0];
    if(hit && hit.object === this.mesh && hit.instanceId != null && this.th[hit.instanceId] > 0){
      this.setHover(hit.instanceId, null); APP.tip.show(APP.tipHTML(DATA[hit.instanceId]), e.clientX, e.clientY);
    } else if(hit && hit.object.userData.block){
      const b = hit.object.userData.block; this.setHover(null, b);
      APP.tip.show('<div class="t">'+b.name.replace(/[<>&]/g,'')+'</div><div>'+b.n+' conjuntos · '+Math.round(100*b.ab/b.n)+' % abandonados</div><div class="m">Faltan '+b.mi.toLocaleString('es-AR')+' actualizaciones prometidas · clic para filtrar</div>', e.clientX, e.clientY);
    } else { this.setHover(null, null); APP.tip.hide(); }
  }
  select(i){ this.sel = i == null ? null : J.get(i); }
  focus(i){
    const j = J.get(i); if(j == null || this.th[j] <= 0) return;
    this.stopSpin(); this.sel = j;
    const x = this.tx[j], z = this.tz[j], h = this.th[j], dir = this.camera.position.clone().sub(this.controls.target).setY(0).normalize();
    const d = 11 + h*.6, p = new THREE.Vector3(x, h*.55, z).add(dir.multiplyScalar(d)).add(new THREE.Vector3(0, 5 + h*.5, 0));
    this.camTo([p.x,p.y,p.z], [x, h*.55, z], 1300);
  }
  frame(){
    const now = performance.now();
    if(this.anim){
      let moving = false;
      for(let j=0;j<N;j++){
        const [x, z, h] = this.cur(j, now);
        if(this.revealed && now - this.t0 - this.dl[j] < 900) moving = true;
        const show = h > .01;
        this.vp.set(x, 0, z); this.vs.set(show ? FOOT : 0, show ? h : 0, show ? FOOT : 0);
        this.m4.compose(this.vp, this.q, this.vs); this.mesh.setMatrixAt(j, this.m4);
      }
      this.mesh.instanceMatrix.needsUpdate = true;
      this.anim = moving || !this.revealed;
    }
    if(this.sel != null && this.th[this.sel] > 0){
      const [x, z, h] = this.cur(this.sel, now);
      this.selBox.visible = h > .02; this.selBox.position.set(x, 0, z); this.selBox.scale.set(FOOT+.08, h+.04, FOOT+.08);
    } else this.selBox.visible = false;
    if(this.tw){
      const k = Math.min(1, (now - this.tw.s)/this.tw.d), e = easeIO(k);
      this.camera.position.lerpVectors(this.tw.p0, this.tw.p1, e); this.controls.target.lerpVectors(this.tw.t0, this.tw.t1, e);
      if(k >= 1) this.tw = null;
    }
    this.controls.update();
    this.renderer.render(this.scene, this.camera);
    this.drawLabels();
  }
  drawLabels(){
    const placed = [], sel = APP.blockSel();
    const order = this.labels.slice().sort((a,b) => (b.b===this.hoverBlock) - (a.b===this.hoverBlock) || b.b.n - a.b.n);
    order.forEach(l => {
      this.v.copy(l.v).project(this.camera);
      let on = this.v.z < 1 && Math.abs(this.v.x) < 1.05 && Math.abs(this.v.y) < 1.05 && (l.b.n >= 6 || l.b === this.hoverBlock || this.group==='th');
      if(on){
        const x = (this.v.x+1)/2*this.w, y = (1-this.v.y)/2*this.h, w = l.el.textContent.length*6.3 + 8;
        const box = {l:x, r:x+w, t:y-9, b:y+9};
        if(placed.some(q => !(box.r<q.l || box.l>q.r || box.b<q.t || box.t>q.b))) on = false;
        else { placed.push(box); l.el.style.transform = 'translate('+x.toFixed(1)+'px,'+y.toFixed(1)+'px) translate(0,-50%)'; }
      }
      l.el.style.opacity = on ? 1 : 0;
      l.el.classList.toggle('on', sel === l.b.name);
    });
  }
}

try {
  const story = new Cloud(document.getElementById('stage3d'), false);
  APP.story = {go:k=>story.go(k)}; story.go(APP.storyStep||0);
  const cloud = new Cloud(document.getElementById('cloud3d'), true);
  APP.cloud = {update:(a,g)=>cloud.update(a,g), focus:i=>cloud.focus(i), select:i=>cloud.select(i)};
  document.querySelectorAll('[data-view]').forEach(b => b.onclick = () => cloud.view(b.dataset.view));
  document.getElementById('zIn').onclick = () => { cloud.stopSpin(); cloud.dolly(.75); };
  document.getElementById('zOut').onclick = () => { cloud.stopSpin(); cloud.dolly(1.33); };
  const spin = document.getElementById('spin');
  if(REDUCED) spin.setAttribute('aria-pressed','false');
  spin.onclick = () => { const on = !cloud.controls.autoRotate; cloud.controls.autoRotate = on; spin.setAttribute('aria-pressed', on); };
  APP.cloudReady();
} catch(e){ console.error(e); APP.cloudFail(); }
try {
  const city = new City(document.getElementById('city3d'));
  APP.city = {update:(a,g)=>city.update(a,g), focus:i=>city.focus(i), select:i=>city.select(i)};
  document.getElementById('cHome').onclick = () => { city.stopSpin(); city.home(); };
  document.getElementById('cIn').onclick = () => { city.stopSpin(); city.dolly(.75); };
  document.getElementById('cOut').onclick = () => { city.stopSpin(); city.dolly(1.33); };
  const cs = document.getElementById('cSpin');
  if(REDUCED) cs.setAttribute('aria-pressed','false');
  cs.onclick = () => { const on = !city.controls.autoRotate; city.controls.autoRotate = on; cs.setAttribute('aria-pressed', on); };
  APP.cityReady();
} catch(e){ console.error(e); APP.cityFail(); }
</script>
</body>
</html>
"""


In [ ]:
# ── Armado del mapa interactivo en HTML ─────────────────────────────────────
payload = {
    "datasets": filas,
    "provincias": [{"n": n, "lat": la, "lon": lo} for n, la, lo, _ in PROVINCIAS],
    "meta": {"generado": AHORA.strftime("%d/%m/%Y"), "total": len(filas)},
}
datos_json = json.dumps(payload, ensure_ascii=False, separators=(",", ":")).replace("</", "<\\/")

html = PLANTILLA_HTML.replace("__DATA__", datos_json)

with open(SALIDA_HTML, "w", encoding="utf-8") as f:
    f.write(html)
df.assign(pv=df.pv.apply("; ".join)).to_csv(SALIDA_CSV, index=False, encoding="utf-8-sig")
print(f"Listo: {SALIDA_HTML} ({len(html)/1e6:.1f} MB) y {SALIDA_CSV}")
print(f"Actualizaciones prometidas que nunca llegaron: {sum(f['mi'] for f in filas):,}")


In [ ]:
# ── Vista previa y descarga ─────────────────────────────────────────────────
VISTA_PREVIA = True

if VISTA_PREVIA:
    from IPython.display import HTML, display
    display(HTML(html))

try:
    from google.colab import files
    files.download(SALIDA_HTML)
    files.download(SALIDA_CSV)
except ImportError:
    print("Fuera de Colab: los archivos quedaron en la carpeta actual.")
